# 08 · Final results, ladders and ablations

Extracted from `image_level1.ipynb` (2 cells) and `image_level2.ipynb` (13 cells). **All outputs preserved.** Originals unmodified.


## A · Two-stream v2 — MODE switch for CV or official


**`IL1` cell 0** —   
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
# TWO-STREAM v2  —  higher resolution, cosine+warmup, EMA, 8-way TTA
#   MODE = "cv"       -> patient-grouped 5-fold      (role_f0..4)
#   MODE = "official" -> official TCIA split         (role_of0..4)
#   Architecture UNCHANGED: mask-weighted pooling + wide stream + aux heads.
#   Every (fold, seed) checkpointed to .npz - a crash costs one fold.
#   >>> RUN WITH QUICK_TEST = True FIRST <<<
# ══════════════════════════════════════════════════════════════════════
import os, gc, time, math, json
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D, LES = "/root/autodl-tmp/CBIS", "mass"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ─────────────────────────── CONFIG ───────────────────────────
QUICK_TEST = False          # <<< True first. One fold, 3 epochs, error check only.
MODE       = "cv"          # "cv" or "official"
TAG        = "twostream_v2"

ST, SW     = 640, 448      # was 512 / 384   <- main gain. Drop to 576/416 if OOM.
BATCH, ACC = 6, 2          # effective batch 12. Lower BATCH if OOM.
SEEDS      = [11, 22]      # [11] for a single-seed run
EPOCHS, FREEZE, WARMUP = 26, 3, 2
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 8e-5    # LR_BACK was 3e-5 - too low
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 8, 2.0
EMA_DECAY  = 0.999
TTA_N      = 8             # was 4
FOLDS      = [0, 1, 2, 3, 4]
USE_AMP    = True
# ──────────────────────────────────────────────────────────────

if QUICK_TEST:
    SEEDS, EPOCHS, FREEZE, WARMUP, MULT, FOLDS = [11], 3, 1, 1, 1, [0]
    print(">>> QUICK TEST: 1 fold, 3 epochs, no checkpoint written\n")

ROLE = "role_f" if MODE == "cv" else "role_of"
CKPT = os.path.join(D, "ckpt_%s_%s" % (TAG, MODE)); os.makedirs(CKPT, exist_ok=True)

WIDE = os.path.join(D, "crops_wide_%s_official" % LES)
if not os.path.isdir(WIDE): WIDE = os.path.join(D, "crops_wide_%s" % LES)
PM   = os.path.join(D, "predmasks_%s_official" % LES)
if not os.path.isdir(PM):   PM = os.path.join(D, "predmasks_%s" % LES)
print("wide crops : %s\npred masks : %s" % (WIDE, PM))

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda p: os.path.basename(str(p)).replace("_img.png", ""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s + "_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s + "_pred.png"))
assert (~d["wimg"].apply(os.path.exists)).sum() == 0, "wide crops missing"
assert "%s0" % ROLE in d.columns, "%s0 column missing" % ROLE

if MODE == "official":
    _te = d["official_split"].astype(str).str.lower().str.contains("test")
    for _k in range(5):
        assert ((d["%s%d" % (ROLE, _k)] == "test") == _te).all(), "fold %d != official test" % _k
    print("verified OFFICIAL SPLIT | train %d | test %d" % ((~_te).sum(), _te.sum()))
else:
    print("patient-grouped 5-fold CV | %d regions | %d patients"
          % (len(d), d.patient_id.nunique()))
print("malignant %.1f%%\n" % (100 * d.label.mean()))

# ─────────────────── cache (RAM ~2 GB at 640/448) ───────────────────
cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
def load(p, size, mask=False):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size, size), np.uint8)
    if im.shape != (size, size):
        im = cv2.resize(im, (size, size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im > 127).astype(np.uint8) if mask else cl.apply(im)

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("cached %d crops in %.0fs (~%.1f GB)"
      % (len(CACHE), time.time() - t0, len(CACHE) * (2*ST*ST + 2*SW*SW) / 1e9))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"]); mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux.keys()); print("helper heads:", meta, "\n")

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug = np.asarray(idx), aug
        self.mult, self.tta = (mult if aug else 1), tta
    def __len__(self): return len(self.idx) * self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand() < .5, np.random.rand() < .5
            kk = np.random.randint(4)
            aff = np.random.rand() < .7
            ang, sc = np.random.uniform(-25, 25), np.random.uniform(.88, 1.14)
            itn = np.random.rand() < .5
            gg, bb = np.random.uniform(.85, 1.15), np.random.uniform(-12, 12)
            def T(im, mk, s):
                if fh: im, mk = im[:, ::-1], mk[:, ::-1]
                if fv: im, mk = im[::-1, :], mk[::-1, :]
                if kk: im, mk = np.rot90(im, kk), np.rot90(mk, kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2, s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s, s), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s, s), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32) * gg + bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta; rot, flip = t % 4, t // 4      # 8 dihedral transforms
            def V(im, mk):
                if rot: im, mk = np.rot90(im, rot), np.rot90(mk, rot)
                if flip: im, mk = im[:, ::-1], mk[:, ::-1]
                return np.ascontiguousarray(im), np.ascontiguousarray(mk)
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        out = []
        for im, mk in [(ti, tm), (wi, wm)]:
            im = np.ascontiguousarray(im).astype(np.float32) / 255.0
            out.append(torch.from_numpy(((np.stack([im]*3, 0) - MEAN) / STD).astype(np.float32)))
            out.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64)
        return out[0], out[1], out[2], out[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, att=2.0):
        super().__init__()
        def bb():
            try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
            except Exception: return models.densenet121(weights=None).features
        self.bt, self.bw, self.att = bb(), bb(), att
        Fd = 1024 * 4
        self.head = nn.Sequential(nn.Linear(Fd, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512, 2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd, 128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[k]))
                                  for k in self.keys])
    def pool(self, b, x, m):
        f = F.relu(b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + self.att * mm
        return torch.cat([(f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6), f.mean((2, 3))], 1)
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([self.pool(self.bt, xt, mt), self.pool(self.bw, xw, mw)], 1)
        return self.head(g), [h(g) for h in self.aux]

class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model):
        for k, v in model.state_dict().items():
            if v.dtype.is_floating_point:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.shadow[k] = v.detach().clone().float()
    def state(self, model):
        ref = model.state_dict()
        return {k: self.shadow[k].to(ref[k].dtype) for k in ref}

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, n_tta=1):
    net.eval(); tot = None
    for t in range(n_tta):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=max(4, BATCH),
                        shuffle=False, num_workers=0, pin_memory=True)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            xt, mt, xw, mw = xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV)
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                o, _ = net(xt, mt, xw, mw)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / n_tta

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    yy = d["label"].values
    n0, n1 = float((yy[tr] == 0).sum()), float((yy[tr] == 1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)

    net = TwoStream(meta, ATT).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for p_ in back: p_.requires_grad = False
    hp = [p_ for n_, p_ in net.named_parameters() if not (n_.startswith("bt.") or n_.startswith("bw."))]

    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD)
    sch, ema = None, None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, bstate, bad, skipped = -1.0, None, 0, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p_ in back: p_.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            T = max(1, EPOCHS - FREEZE)
            def lam(e):
                if e < WARMUP: return (e + 1) / WARMUP
                return 0.5 * (1 + math.cos(math.pi * (e - WARMUP) / max(1, T - WARMUP)))
            sch = torch.optim.lr_scheduler.LambdaLR(opt, lam)
            ema = EMA(net, EMA_DECAY)

        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        opt.zero_grad(set_to_none=True)
        for bi, (xt, mt, xw, mw, t_, a_) in enumerate(tl):
            xt, mt, xw, mw = (xt.to(DEV, non_blocking=True), mt.to(DEV, non_blocking=True),
                              xw.to(DEV, non_blocking=True), mw.to(DEV, non_blocking=True))
            t_, a_ = t_.to(DEV), a_.to(DEV)
            with torch.amp.autocast(device_type="cuda", enabled=USE_AMP):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(gg.float(), a_[:, h], ignore_index=-1)
                        for h, gg in enumerate(ax)) / len(ax)
                loss = loss / ACC
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward()
            if (bi + 1) % ACC == 0:
                scaler.unscale_(opt)
                gn = torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                if torch.isfinite(gn): scaler.step(opt)
                else: skipped += 1
                scaler.update(); opt.zero_grad(set_to_none=True)
                if ema is not None: ema.update(net)
        if sch is not None: sch.step()

        # evaluate the EMA weights once they exist
        if ema is not None:
            raw = {k: v.detach().clone() for k, v in net.state_dict().items()}
            net.load_state_dict(ema.state(net))
        pv = predict(net, va, n_tta=1)
        if not np.all(np.isfinite(pv)):
            print("      ep %2d  NON-FINITE predictions - diverged" % ep)
            if bstate is not None: break
            raise RuntimeError("diverged; set USE_AMP = False and rerun")
        auc = roc_auc_score(yy[va], pv) if len(set(yy[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad = auc, 0
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}; star = " *"
        else: bad += 1
        if ema is not None: net.load_state_dict(raw)
        print("      ep %2d  val-AUC %.4f%s" % (ep, auc, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break

    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, n_tta=TTA_N)
    if skipped: print("      (%d optimiser steps skipped on non-finite grads)" % skipped)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, best

# ─────────────────────────── run ───────────────────────────
y = d["label"].values
acc_p, cnt_p = np.zeros(len(d)), np.zeros(len(d))
oof = np.full(len(d), np.nan)

for k in FOLDS:
    role = d["%s%d" % (ROLE, k)]
    tr = np.where(role == "train")[0]; va = np.where(role == "val")[0]; te = np.where(role == "test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK train/test"
    assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "LEAK val/test"
    print("\n### fold %d | train %d val %d test %d" % (k, len(tr), len(va), len(te)))
    ps, t0 = [], time.time()
    for sd in SEEDS:
        ck = os.path.join(CKPT, "f%d_s%d.npz" % (k, sd))
        if os.path.exists(ck) and not QUICK_TEST:
            z = np.load(ck, allow_pickle=True)
            if list(z["te_idx"]) == list(te):
                ps.append(z["prob"]); print("    seed %d: loaded checkpoint (val %.4f)" % (sd, float(z["val"])))
                continue
            print("    seed %d: stale checkpoint, retraining" % sd)
        p_, bv = train_one(tr, va, te, sd); ps.append(p_)
        if not QUICK_TEST:
            np.savez(ck, prob=p_, val=bv, te_idx=np.array(te))
        print("    seed %d: best val %.4f | fold test AUC %.4f" % (sd, bv, roc_auc_score(y[te], p_)))
    pmn = np.mean(ps, 0)
    acc_p[te] += pmn; cnt_p[te] += 1; oof[te] = acc_p[te] / cnt_p[te]
    print("  FOLD %d AUC %.4f | pooled so far %.4f (%.0fs)"
          % (k, roc_auc_score(y[te], pmn), roc_auc_score(y[te], oof[te]), time.time() - t0))

done = ~np.isnan(oof)
res = d.loc[done, ["img", "lesion_key", "label"]].copy(); res["prob"] = oof[done]
Lg = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean")).reset_index()
print("\n" + "=" * 70)
print("TWO-STREAM v2   MODE=%s   ST=%d SW=%d   seeds=%s" % (MODE, ST, SW, SEEDS))
print("=" * 70)
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(res.label, res.prob), len(res)))
print("  per-lesion AUC %.4f  (n=%d)" % (roc_auc_score(Lg.y, Lg.p), len(Lg)))
print("  reference: v1 official image 0.8769 / lesion 0.9043 | v1 CV image 0.8734 / lesion 0.8885")
if not QUICK_TEST:
    suffix = "officialsplit" if MODE == "official" else "cv"
    out = os.path.join(D, "cv_%s_%s_%s_oof.csv" % (LES, TAG, suffix))
    res.rename(columns={"label": "true"}).to_csv(out, index=False)
    print("\n  saved %s" % os.path.basename(out))
else:
    print("\n  QUICK TEST clean. Set QUICK_TEST = False and rerun.")

wide crops : /root/autodl-tmp/CBIS/crops_wide_mass_official
pred masks : /root/autodl-tmp/CBIS/predmasks_mass_official
patient-grouped 5-fold CV | 1696 regions | 892 patients
malignant 46.2%

cached 1696 crops in 20s (~2.1 GB)
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5} 


### fold 0 | train 1054 val 264 test 378
      ep  1  val-AUC 0.7412 *
      ep  2  val-AUC 0.7703 *
      ep  3  val-AUC 0.7283
      ep  4  val-AUC 0.7070
      ep  5  val-AUC 0.7485
      ep  6  val-AUC 0.7909 *
      ep  7  val-AUC 0.8279 *
      ep  8  val-AUC 0.8586 *
      ep  9  val-AUC 0.8797 *
      ep 10  val-AUC 0.8857 *
      ep 11  val-AUC 0.8916 *
      ep 12  val-AUC 0.9007 *
      ep 13  val-AUC 0.9056 *
      ep 14  val-AUC 0.9080 *
      ep 15  val-AUC 0.9084 *
      ep 16  val-AUC 0.9061
      ep 17  val-AUC 0.9002
      ep 18  val-AUC 0.8965
      ep 19  val-AUC 0.8942
      ep 20  val-AUC 0.8912
      ep 21  val-AUC 0.8883
      ep 22  val-AUC 0.8861
      ep 23  val-AUC 0.

## B · Mask provenance — official masks vs CV masks


**`IL1` cell 1** — import os, numpy as np, pandas as pd  
<sub>1 output block(s) preserved</sub>


In [1]:
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
if "side" not in d.columns:
    fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
    d=d.merge(fx[["_k","side"]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int); d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
B=d.set_index("_k")

for fn in ("cv_mass_twostream_officialsplit_oof.csv",
           "cv_mass_twostream_officialsplit_oof_cvmasks.csv"):
    f=os.path.join(D,fn)
    if not os.path.exists(f): print("MISSING %s"%fn); continue
    m=pd.read_csv(f); m["_k"]=m["img"].map(stem); m=m[m["_k"].isin(B.index)]
    row=[fn.replace("cv_mass_twostream_officialsplit_oof","").replace(".csv","") or "  (standard)"]
    for nm,key in (("ROI",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")):
        t=pd.DataFrame(dict(p=m.prob.values,y=B.loc[m._k,"y"].values,
                            k=(m._k.values if key is None else B.loc[m._k,key].values)))
        g=t.groupby("k").agg(p=("p","mean"),y=("y","max"))
        row.append("%s %.4f"%(nm,roc_auc_score(g.y,g.p)))
    print("  ".join(row))
print("\nIf the two lines are close, mask provenance is not a problem.")
print("If _cvmasks is clearly lower, the standard file used masks from a segmentation")
print("model that saw the official-test patients, and _cvmasks is what you must report.")

  (standard)  ROI 0.8769  LESION 0.9043  BREAST 0.9016  PATIENT 0.9041
_cvmasks  ROI 0.8695  LESION 0.9004  BREAST 0.8948  PATIENT 0.8967

If the two lines are close, mask provenance is not a problem.
If _cvmasks is clearly lower, the standard file used masks from a segmentation
model that saw the official-test patients, and _cvmasks is what you must report.


## C · Final results on the official split


**`IL2` cell 0** —   
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 4 — RAISE THE SCORE, OFFICIAL SPLIT
#
#   PROTOCOL (unchanged, locked): fit on official TRAIN (1,318 img / 782 les),
#   report on official TEST (378 img / 223 les). Zero patient overlap.
#
#   Every candidate score is judged ONLY by its TRAIN AUC. The test set is
#   touched once, at the end, by the single winning candidate.
#   Tie-break rule: among candidates within 0.002 TRAIN AUC of the best,
#   take the SIMPLEST (fewest members). Guards against selection overfitting.
#
#   No GPU, no training. ~2 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, glob, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

D          = "/root/autodl-tmp/CBIS"
SENS_FLOOR = 0.70
GRID       = np.round(np.arange(0.02, 0.99, 0.01), 3)
MIN_N, MIN_POS, PASSES = 20, 3, 12
TIE, MAXK  = 0.002, 4
NBOOT, RNG = 2000, np.random.default_rng(7)
MATCH_SENS = [0.80, 0.85, 0.90, 0.95]

stem  = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
LOGIT = lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))
SIG   = lambda z: 1/(1+np.exp(-z))

# ─────────────────────────────────────────────────────────────────────────
# PART A — discover every (official-train, official-test) prediction pair
# ─────────────────────────────────────────────────────────────────────────
print("="*80); print("PART A — AVAILABLE MEMBERS (train scores are out-of-fold)"); print("="*80)

d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv"))
d["_k"] = d["img"].map(stem); assert d["_k"].is_unique
d["y"]  = d["label"].astype(int)
d["a"]  = pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"] = np.where(d["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
META = d.set_index("_k")[["y","a","lesion_key","patient_id","sp"]]

def load(fn):
    f = os.path.join(D, fn)
    if not os.path.exists(f): return None
    m = pd.read_csv(f)
    if "img" not in m.columns: return None
    pc = "prob" if "prob" in m.columns else None
    if pc is None:
        c = [x for x in m.columns if m[x].dtype.kind=="f" and m[x].between(0,1).all()]
        if not c: return None
        pc = c[0]
    m["_k"] = m["img"].map(stem)
    m = m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")
    return m[m["_k"].isin(META.index)]

MEM = {}
for trf in sorted(glob.glob(os.path.join(D, "*officialtrain*.csv"))):
    b   = os.path.basename(trf)
    tef = b.replace("officialtrain", "officialsplit")
    A, B = load(b), load(tef)
    if A is None or B is None or len(A) < 900 or len(B) < 300: continue
    nm = b.replace("cv_mass_","").replace("_officialtrain_oof","").replace(".csv","")
    ka = A.set_index("_k")["p"]; kb = B.set_index("_k")["p"]
    ya = META.loc[ka.index,"y"].values; yb = META.loc[kb.index,"y"].values
    if len(np.unique(ya))<2 or len(np.unique(yb))<2: continue
    atr, ate = roc_auc_score(ya, ka.values), roc_auc_score(yb, kb.values)
    if atr > 0.97: print("   SKIP %-34s TRAIN AUC %.4f (leakage guard)" % (nm, atr)); continue
    MEM[nm] = (ka, kb)
    print("   %-34s n_tr %4d n_te %3d   TRAIN AUC %.4f   test %.4f"
          % (nm, len(ka), len(kb), atr, ate))
assert MEM, "no paired official train/test prediction files found"

# ─────────────────────────────────────────────────────────────────────────
# PART B — is the two-stream file already seed-averaged?
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "="*80); print("PART B — SEED CHECK"); print("="*80)
one, avg = load("cv_mass_twostream_officialsplit_oof_1seed.csv"), load("cv_mass_twostream_officialsplit_oof.csv")
if one is not None and avg is not None:
    j = one.merge(avg, on="_k", suffixes=("_1","_m"))
    same = np.allclose(j.p_1.values, j.p_m.values, atol=1e-6)
    print("  officialsplit_oof vs _1seed : %s"
          % ("IDENTICAL -> current file is SINGLE seed, averaging is still available"
             if same else "DIFFERENT -> current file is already a seed average"))
    print("  corr %.5f | mean abs diff %.5f"
          % (np.corrcoef(j.p_1, j.p_m)[0,1], np.abs(j.p_1-j.p_m).mean()))
else:
    print("  one of the two files is missing; skipping")

s11, s22 = load("cv_mass_twostream_s11.csv"), load("cv_mass_twostream_s22.csv")
if s11 is not None and s22 is not None:
    j = s11.merge(s22, on="_k", suffixes=("_a","_b"))
    yy = META.loc[j._k,"y"].values
    print("  CV seeds: s11 AUC %.4f | s22 AUC %.4f | average %.4f  (%+.4f)"
          % (roc_auc_score(yy,j.p_a), roc_auc_score(yy,j.p_b),
             roc_auc_score(yy,(j.p_a+j.p_b)/2),
             roc_auc_score(yy,(j.p_a+j.p_b)/2)-max(roc_auc_score(yy,j.p_a),roc_auc_score(yy,j.p_b))))

# ─────────────────────────────────────────────────────────────────────────
# PART C — build candidates, judge them on TRAIN ONLY
# ─────────────────────────────────────────────────────────────────────────
print("\n" + "="*80); print("PART C — CANDIDATE SELECTION (TRAIN AUC ONLY)"); print("="*80)

names = list(MEM)
ktr = sorted(set.intersection(*[set(MEM[n][0].index) for n in names]))
kte = sorted(set.intersection(*[set(MEM[n][1].index) for n in names]))
Ytr, Yte = META.loc[ktr,"y"].values, META.loc[kte,"y"].values
Ptr = {n: MEM[n][0].loc[ktr].values for n in names}
Pte = {n: MEM[n][1].loc[kte].values for n in names}
print("  common rows: train %d, test %d" % (len(ktr), len(kte)))

def mix(sel, P, how):
    M = np.column_stack([P[m] for m in sel])
    return M.mean(1) if how=="prob" else SIG(LOGIT(M).mean(1))

CAND = []
for n in names:                                   # singles
    CAND.append(((n,), "prob", roc_auc_score(Ytr, Ptr[n])))
for how in ("prob","logit"):                      # greedy forward, capped
    ch, best = [], -np.inf
    while len(ch) < MAXK:
        pk, pa = None, best
        for n in names:
            if n in ch: continue
            v = roc_auc_score(Ytr, mix(ch+[n], Ptr, how))
            if v > pa + 1e-6: pk, pa = n, v
        if pk is None: break
        ch.append(pk); best = pa
        CAND.append((tuple(ch), how, best))

CAND.sort(key=lambda c: -c[2])
top = CAND[0][2]
print("\n  top candidates by TRAIN AUC")
for sel, how, v in CAND[:8]:
    print("    %-6s %.4f  [%s]" % (how, v, " + ".join(sel)))
elig = [c for c in CAND if c[2] >= top - TIE]
SEL, HOW, SAUC = min(elig, key=lambda c: (len(c[0]), -c[2]))
print("\n  within %.3f of best: %d candidates -> take the simplest" % (TIE, len(elig)))
print("  CHOSEN: %s  [%s]   TRAIN AUC %.4f" % (HOW, " + ".join(SEL), SAUC))

# ─────────────────────────────────────────────────────────────────────────
# PART D — decision layer + final report on TEST
# ─────────────────────────────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,floor=SENS_FLOOR):
    P,N = int(y.sum()), len(y)
    tp,fp = _cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se = (tp+(N-P)-fp)/N, tp/max(P,1); ok = se>=floor-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[int(np.argmax(se))])
def _asc(y,p,a,init,floor):
    N,P = len(y), int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1) < floor-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=floor-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j] > (TP+TN)/N + 1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,floor=SENS_FLOOR):
    g0=fit_global(y,p,floor); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (0.50,g0,0.35,0.45,0.55,0.65,max(.02,g0-.1),min(.98,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.2,.8)) for g in cats} for _ in range(4)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,floor)
        if ac>ba+1e-12: best,ba=t,ac
    return best,g0
ap = lambda p,a,t,g0: (p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),
                sens=tp/max(tp+fn,1),spec=tn/max(tn+fp,1),fn=fn)
def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)), float(np.percentile(o,97.5))

BASE = "twostream" if "twostream" in names else names[0]
str_, ste_ = mix(SEL,Ptr,HOW), mix(SEL,Pte,HOW)
FINAL = {}
for level in ("IMAGE","LESION"):
    if level=="IMAGE":
        ytr,atr,ptr = Ytr, META.loc[ktr,"a"].values, str_
        yte,ate,pte = Yte, META.loc[kte,"a"].values, ste_
        btr,bte = Ptr[BASE], Pte[BASE]
        pat = META.loc[kte,"patient_id"].values
    else:
        def agg(keys,p):
            t=pd.DataFrame(dict(lk=META.loc[keys,"lesion_key"].values, y=META.loc[keys,"y"].values,
                                a=META.loc[keys,"a"].values, pid=META.loc[keys,"patient_id"].values, p=p))
            g=t.groupby("lk",sort=True)
            return (g.p.mean().values, g.y.max().values, g.a.first().values, g.pid.first().values)
        ptr,ytr,atr,_   = agg(ktr,str_);  btr = agg(ktr,Ptr[BASE])[0]
        pte,yte,ate,pat = agg(kte,ste_);  bte = agg(kte,Pte[BASE])[0]

    gt_b = fit_global(ytr,btr); gt_s = fit_global(ytr,ptr)
    tb,g0b = fit_bir(ytr,btr,atr); ts,g0s = fit_bir(ytr,ptr,atr)
    rows=[("OLD  two-stream + one threshold", bte,(bte>=gt_b).astype(int)),
          ("OLD  two-stream + Novelty 2",     bte, ap(bte,ate,tb,g0b)),
          ("NEW  selected score + one threshold", pte,(pte>=gt_s).astype(int)),
          ("NEW  selected score + Novelty 2  <== FINAL", pte, ap(pte,ate,ts,g0s))]
    print("\n"+"="*80)
    print("%s LEVEL — OFFICIAL TEST, n=%d (%.1f%% malignant)" % (level,len(yte),100*yte.mean()))
    print("="*80)
    print("  %-44s %-18s %-16s %-6s %-6s %s" % ("system","AUC [95% CI]","acc [95% CI]","sens","spec","missed"))
    for nm,sc,yh in rows:
        m=M(yte,sc,yh); la,ha=ci(yte,sc,yh,pat,"auc"); lc,hc=ci(yte,sc,yh,pat,"acc")
        print("  %-44s %.4f[%.3f-%.3f] %.1f%%[%.1f-%.1f] %.3f  %.3f  %d"
              % (nm,m["auc"],la,ha,100*m["acc"],100*lc,100*hc,m["sens"],m["spec"],m["fn"]))
        FINAL["%s | %s"%(level,nm)] = m
    print("\n  thresholds: %s (global %.2f)" % ({int(k):round(v,2) for k,v in sorted(ts.items())}, gt_s))
    print("\n  SPECIFICITY AT MATCHED SENSITIVITY  (refit on TRAIN at each target)")
    print("    %-8s %-24s %-24s %s" % ("target","one threshold","per-BI-RADS","gain"))
    for s in MATCH_SENS:
        g_s=fit_global(ytr,ptr,s); t_s,g0_=fit_bir(ytr,ptr,atr,s)
        m1=M(yte,pte,(pte>=g_s).astype(int)); m2=M(yte,pte,ap(pte,ate,t_s,g0_))
        print("    %-8.2f sens %.3f spec %.3f     sens %.3f spec %.3f     %+.3f"
              % (s,m1["sens"],m1["spec"],m2["sens"],m2["spec"],m2["spec"]-m1["spec"]))

with open(os.path.join(D,"FINAL_v2.json"),"w") as f:
    json.dump({"chosen":list(SEL),"how":HOW,"train_auc":SAUC,"results":FINAL}, f, indent=2, default=float)
print("\nsaved FINAL_v2.json")

PART A — AVAILABLE MEMBERS (train scores are out-of-fold)
   endtoend_fused                     n_tr 1318 n_te 378   TRAIN AUC 0.8398   test 0.8192
   endtoend                           n_tr 1318 n_te 378   TRAIN AUC 0.8442   test 0.8194
   handcrafted                        n_tr 1318 n_te 378   TRAIN AUC 0.6973   test 0.6781
   imageonly                          n_tr 1318 n_te 378   TRAIN AUC 0.7900   test 0.7994
   twostream_calcpre                  n_tr 1318 n_te 378   TRAIN AUC 0.8789   test 0.8582
   twostream_calcpre_cvmasks          n_tr 1318 n_te 378   TRAIN AUC 0.8811   test 0.8660
   twostream                          n_tr 1318 n_te 378   TRAIN AUC 0.8974   test 0.8769
   twostream_1seed                    n_tr 1318 n_te 378   TRAIN AUC 0.8974   test 0.8769

PART B — SEED CHECK
  officialsplit_oof vs _1seed : IDENTICAL -> current file is SINGLE seed, averaging is still available
  corr 1.00000 | mean abs diff 0.00000
  CV seeds: s11 AUC 0.8553 | s22 AUC 0.8713 | average 0.873

**`IL2` cell 1** —   
<sub>1 output block(s) preserved</sub>


In [3]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 5 — FINAL. OFFICIAL SPLIT, SENSITIVITY FLOOR 0.90
#
#   PROTOCOL (locked): fit everything on official TRAIN (1,318 img / 782 les),
#   freeze, report once on official TEST (378 img / 223 les). 0 patient overlap.
#   Train scores : cv_mass_twostream_officialtrain_oof.csv  (out-of-fold INSIDE
#                  official train — no official-test patient ever seen)
#   Test scores  : cv_mass_twostream_officialsplit_oof.csv
#
#   S1 CNN + one threshold      S2 CNN + per-BI-RADS   [NOVELTY 2]
#   S3 fusion + one threshold   S4 fusion + per-BI-RADS
#   All four honour the SAME sensitivity floor, so the comparison is fair.
#
#   No GPU, no training. ~3 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, json, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

D        = "/root/autodl-tmp/CBIS"
TRF      = "cv_mass_twostream_officialtrain_oof.csv"
TEF      = "cv_mass_twostream_officialsplit_oof.csv"
FLOOR    = 0.90                                   # <-- the clinical specification
SWEEP    = [0.70,0.75,0.80,0.85,0.88,0.90,0.92,0.95]
GRID     = np.round(np.arange(0.01,0.995,0.005),3)
MIN_N, MIN_POS, PASSES = 20, 3, 15
NBOOT, RNG = 2000, np.random.default_rng(7)

stem  = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
LOGIT = lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))

# ─── PART 1 : data ────────────────────────────────────────────────────────
print("="*80); print("PART 1 — INPUTS"); print("="*80)
d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
assert d["_k"].is_unique
for c in ("density","subtlety"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d["label"].astype(int)
d["a"]=pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["ds"]=pd.to_numeric(d["density"],errors="coerce");  d["ds"]=d["ds"].fillna(d["ds"].median())
d["sb"]=pd.to_numeric(d["subtlety"],errors="coerce"); d["sb"]=d["sb"].fillna(d["sb"].median())

def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); pc="prob" if "prob" in m.columns else \
      [c for c in m.columns if m[c].dtype.kind=="f" and m[c].between(0,1).all()][0]
    m["_k"]=m["img"].map(stem); return m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")

TR = d.merge(load(TRF),on="_k").reset_index(drop=True)
TE = d.merge(load(TEF),on="_k").reset_index(drop=True)
assert not (set(TR.patient_id) & set(TE.patient_id)), "patient overlap"
print("  TRAIN %4d img  %3d les  %3d pat  %.1f%% malig   (%s)"
      % (len(TR),TR.lesion_key.nunique(),TR.patient_id.nunique(),100*TR.y.mean(),TRF))
print("  TEST  %4d img  %3d les  %3d pat  %.1f%% malig   (%s)"
      % (len(TE),TE.lesion_key.nunique(),TE.patient_id.nunique(),100*TE.y.mean(),TEF))
print("  patient overlap 0  OK   |  sensitivity floor %.2f" % FLOOR)

def lesion(t):
    g=t.groupby("lesion_key",sort=True)
    return g.agg(p=("p","mean"),y=("y","max"),a=("a","first"),ds=("ds","first"),
                 sb=("sb","first"),patient_id=("patient_id","first")).reset_index()

# ─── PART 2 : machinery ───────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)

def fit_global(y,p,floor):
    """single threshold: best accuracy among cuts that meet the floor"""
    P,N=int(y.sum()),len(y)
    tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N, tp/max(P,1); ok=se>=floor-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])

def _asc(y,p,a,init,floor):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1) < floor-1e-12: return tau,-1.0,el          # infeasible start
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=floor-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j] > (TP+TN)/N + 1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N,el

def fit_bir(y,p,a,floor):
    """multi-start coordinate ascent; g0 start is feasible by construction"""
    g0=fit_global(y,p,floor); cats=[int(c) for c in np.unique(a)]
    fixed=[g0,0.05,0.15,0.25,0.35,0.45,0.50,0.55,0.65,max(0.01,g0-0.10),min(0.99,g0+0.10)]
    st=[{g:v for g in cats} for v in fixed]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba,bel=None,-np.inf,[]
    for s in st:
        t,ac,el=_asc(y,p,a,s,floor)
        if ac>ba+1e-12: best,ba,bel=t,ac,el
    if best is None or ba<0: best={g:g0 for g in cats}          # fall back to global
    return best,g0,bel,ba

ap = lambda p,a,t,g0: (p>=np.array([t.get(int(g),g0) for g in a])).astype(int)

def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),sens=tp/max(tp+fn,1),
                spec=tn/max(tn+fp,1),fp=fp,fn=fn)

def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))

CATS=[0,1,2,3,4,5]
def fuse(tr,te):
    X=lambda t: np.column_stack([LOGIT(t.p.values)]+
        [(t.a.values==g).astype(float) for g in CATS]+[t.ds.values,t.sb.values])
    Xtr,Xte=X(tr),X(te); mu,sd=Xtr.mean(0),Xtr.std(0)+1e-9
    lr=LogisticRegression(C=0.05,max_iter=5000).fit((Xtr-mu)/sd,tr.y.values)
    return lr.predict_proba((Xtr-mu)/sd)[:,1], lr.predict_proba((Xte-mu)/sd)[:,1]

# ─── PART 3 : primary result at FLOOR ─────────────────────────────────────
ROWS, FINAL = [], {}
for level,ftr,fte in (("IMAGE",TR,TE), ("LESION",lesion(TR),lesion(TE))):
    ytr,ptr,atr = ftr.y.values.astype(int), ftr.p.values, ftr.a.values.astype(int)
    yte,pte,ate = fte.y.values.astype(int), fte.p.values, fte.a.values.astype(int)
    pat = fte.patient_id.values
    ztr,zte = fuse(ftr,fte)

    print("\n"+"="*80)
    print("%s LEVEL — OFFICIAL TEST, n=%d (%.1f%% malignant, %d patients) | FLOOR %.2f"
          % (level,len(yte),100*yte.mean(),len(np.unique(pat)),FLOOR))
    print("  thresholds and fusion fitted on official TRAIN n=%d, then frozen" % len(ytr))
    print("="*80)

    g1 = fit_global(ytr,ptr,FLOOR)
    t2,g2,el2,tr2 = fit_bir(ytr,ptr,atr,FLOOR)
    g3 = fit_global(ytr,ztr,FLOOR)
    t4,g4,el4,_   = fit_bir(ytr,ztr,atr,FLOOR)
    sysd=[("S1  CNN + one threshold",              pte,(pte>=g1).astype(int)),
          ("S2  CNN + per-BI-RADS   [NOVELTY 2]",  pte, ap(pte,ate,t2,g2)),
          ("S3  fusion + one threshold",           zte,(zte>=g3).astype(int)),
          ("S4  fusion + per-BI-RADS",             zte, ap(zte,ate,t4,g4))]
    print("  %-40s %-18s %-16s %-6s %-6s %s"
          % ("system","AUC [95% CI]","acc [95% CI]","sens","spec","missed"))
    got={}
    for nm,sc,yh in sysd:
        m=M(yte,sc,yh); la,ha=ci(yte,sc,yh,pat,"auc"); lc,hc=ci(yte,sc,yh,pat,"acc")
        got[nm[:2]]=m; FINAL["%s|%s"%(level,nm[:2])]=dict(m,auc_ci=[la,ha],acc_ci=[lc,hc])
        star="  <== FINAL" if nm.startswith("S2") else ""
        print("  %-40s %.4f[%.3f-%.3f] %.1f%%[%.1f-%.1f] %.3f  %.3f  %d%s"
              % (nm,m["auc"],la,ha,100*m["acc"],100*lc,100*hc,m["sens"],m["spec"],m["fn"],star))

    print("\n  frozen thresholds  %s"
          % {int(k):round(v,3) for k,v in sorted(t2.items())})
    print("  categories with their own threshold: %s   (others use global %.3f)"
          % (el2,g2))
    print("  train accuracy of the fitted layer: %.4f" % tr2)
    print("\n  S2 vs S1  (your layer vs one cut, same score, same floor) : %+.2f acc pts, %+d cancers found"
          % (100*(got["S2"]["acc"]-got["S1"]["acc"]), got["S1"]["fn"]-got["S2"]["fn"]))
    print("  S2 vs S3  (your layer vs radiologist-assessment fusion)    : %+.2f acc pts, %+d cancers found"
          % (100*(got["S2"]["acc"]-got["S3"]["acc"]), got["S3"]["fn"]-got["S2"]["fn"]))
    print("  S4 vs S3  (redundancy: layer on top of BI-RADS fusion)     : %+.2f acc pts"
          % (100*(got["S4"]["acc"]-got["S3"]["acc"])))

    # ─── PART 4 : floor sweep ─────────────────────────────────────────────
    print("\n  FLOOR SWEEP — everything refitted on TRAIN at each floor")
    print("    %-6s %-24s %-24s %s" % ("floor","S1 one threshold","S2 per-BI-RADS","gain"))
    for fl in SWEEP:
        ga=fit_global(ytr,ptr,fl); tb,gb,_,_=fit_bir(ytr,ptr,atr,fl)
        m1=M(yte,pte,(pte>=ga).astype(int)); m2=M(yte,pte,ap(pte,ate,tb,gb))
        ROWS.append(dict(level=level,floor=fl,
                         s1_acc=m1["acc"],s1_sens=m1["sens"],s1_spec=m1["spec"],s1_fn=m1["fn"],
                         s2_acc=m2["acc"],s2_sens=m2["sens"],s2_spec=m2["spec"],s2_fn=m2["fn"]))
        print("    %-6.2f acc %5.1f%% se %.3f sp %.3f   acc %5.1f%% se %.3f sp %.3f   %+.2f pts, %+d cancers"
              % (fl,100*m1["acc"],m1["sens"],m1["spec"],
                 100*m2["acc"],m2["sens"],m2["spec"],
                 100*(m2["acc"]-m1["acc"]), m1["fn"]-m2["fn"]))

pd.DataFrame(ROWS).to_csv(os.path.join(D,"floor_sweep_official.csv"),index=False)
with open(os.path.join(D,"FINAL_floor090.json"),"w") as f:
    json.dump(dict(protocol="official TCIA split; fit on train, frozen; report on test",
                   floor=FLOOR,train_scores=TRF,test_scores=TEF,results=FINAL),
              f,indent=2,default=float)
print("\nsaved floor_sweep_official.csv  and  FINAL_floor090.json")

PART 1 — INPUTS
  TRAIN 1318 img  782 les  691 pat  48.3% malig   (cv_mass_twostream_officialtrain_oof.csv)
  TEST   378 img  223 les  201 pat  38.9% malig   (cv_mass_twostream_officialsplit_oof.csv)
  patient overlap 0  OK   |  sensitivity floor 0.90

IMAGE LEVEL — OFFICIAL TEST, n=378 (38.9% malignant, 201 patients) | FLOOR 0.90
  thresholds and fusion fitted on official TRAIN n=1318, then frozen
  system                                   AUC [95% CI]       acc [95% CI]     sens   spec   missed
  S1  CNN + one threshold                  0.8769[0.825-0.919] 75.7%[70.7-80.5] 0.884  0.675  17
  S2  CNN + per-BI-RADS   [NOVELTY 2]      0.8769[0.828-0.918] 81.0%[76.3-85.4] 0.878  0.766  18  <== FINAL
  S3  fusion + one threshold               0.9061[0.861-0.942] 79.1%[73.9-84.1] 0.884  0.732  17
  S4  fusion + per-BI-RADS                 0.9061[0.866-0.943] 80.2%[75.1-84.9] 0.884  0.749  17

  frozen thresholds  {0: 0.515, 1: 0.44, 2: 0.44, 3: 0.495, 4: 0.455, 5: 0.01}
  categories with t

**`IL2` cell 2** —   
<sub>1 output block(s) preserved</sub>


In [5]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 6 — COMPLETE OPERATING GRID, OFFICIAL TEST SET   (fixed)
#
#   4 systems  x  8 sensitivity floors  x  4 reporting units
#   Everything fitted on official TRAIN and frozen. Test touched for scoring only.
#
#   units : IMAGE 378 | LESION 223 | BREAST ~210 | PATIENT 201
#   S1 CNN + one threshold     S2 CNN + per-BI-RADS  [NOVELTY 2]
#   S3 fusion + one threshold  S4 fusion + per-BI-RADS
#
#   No GPU, no training. ~4 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, json, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

D     = "/root/autodl-tmp/CBIS"
TRF   = "cv_mass_twostream_officialtrain_oof.csv"
TEF   = "cv_mass_twostream_officialsplit_oof.csv"
SWEEP = [0.70,0.75,0.80,0.85,0.88,0.90,0.92,0.95]
GRID  = np.round(np.arange(0.01,0.995,0.005),3)
MIN_N, MIN_POS, PASSES = 20, 3, 15
NBOOT, RNG = 2000, np.random.default_rng(7)

stem  = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
LOGIT = lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))

# ─── data ─────────────────────────────────────────────────────────────────
d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
assert d["_k"].is_unique
for c in ("density","subtlety","side"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d["label"].astype(int)
d["a"]=pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["ds"]=pd.to_numeric(d["density"],errors="coerce");  d["ds"]=d["ds"].fillna(d["ds"].median())
d["sb"]=pd.to_numeric(d["subtlety"],errors="coerce"); d["sb"]=d["sb"].fillna(d["sb"].median())
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)

def load(fn):
    m=pd.read_csv(os.path.join(D,fn))
    pc="prob" if "prob" in m.columns else [c for c in m.columns
        if m[c].dtype.kind=="f" and m[c].between(0,1).all()][0]
    m["_k"]=m["img"].map(stem); return m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")

TR=d.merge(load(TRF),on="_k").reset_index(drop=True)
TE=d.merge(load(TEF),on="_k").reset_index(drop=True)
assert not (set(TR.patient_id)&set(TE.patient_id))

def roll(t,key):
    """aggregate to a reporting unit; never re-aggregate the grouping column"""
    if key is None: return t.reset_index(drop=True)
    spec = dict(p=("p","mean"), y=("y","max"), a=("a","max"),
                ds=("ds","first"), sb=("sb","first"))
    if key != "patient_id":
        spec["patient_id"] = ("patient_id","first")
    out = t.groupby(key, sort=True).agg(**spec).reset_index()
    if "patient_id" not in out.columns:
        out["patient_id"] = out[key].values
    return out

UNITS=[("IMAGE",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")]

# ─── machinery ────────────────────────────────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TP+TN)/N+1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,0.05,0.15,0.25,0.35,0.45,0.50,0.55,0.65,
                                      max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),sens=tp/max(tp+fn,1),
                spec=tn/max(tn+fp,1),fn=fn)
def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))
CATS=[0,1,2,3,4,5]
def fuse(tr,te):
    X=lambda t: np.column_stack([LOGIT(t.p.values)]+
        [(t.a.values==g).astype(float) for g in CATS]+[t.ds.values,t.sb.values])
    Xtr,Xte=X(tr),X(te); mu,sd=Xtr.mean(0),Xtr.std(0)+1e-9
    lr=LogisticRegression(C=0.05,max_iter=5000).fit((Xtr-mu)/sd,tr.y.values)
    return lr.predict_proba((Xtr-mu)/sd)[:,1],lr.predict_proba((Xte-mu)/sd)[:,1]

# ─── the grid ─────────────────────────────────────────────────────────────
ALL=[]
for uname,key in UNITS:
    ftr,fte=roll(TR,key),roll(TE,key)
    ytr,ptr,atr=ftr.y.values.astype(int),ftr.p.values,ftr.a.values.astype(int)
    yte,pte,ate=fte.y.values.astype(int),fte.p.values,fte.a.values.astype(int)
    ztr,zte=fuse(ftr,fte)
    print("\n"+"="*92)
    print("%s LEVEL — OFFICIAL TEST n=%d (%.1f%% malignant) | train n=%d | AUC cnn %.4f  fusion %.4f"
          % (uname,len(yte),100*yte.mean(),len(ytr),
             roc_auc_score(yte,pte),roc_auc_score(yte,zte)))
    print("="*92)
    print("  %-6s %-22s %-22s %-22s %-22s"
          % ("floor","S1 cnn+one","S2 cnn+BI-RADS [YOURS]","S3 fus+one","S4 fus+BI-RADS"))
    for fl in SWEEP:
        g1=fit_global(ytr,ptr,fl); t2,g2=fit_bir(ytr,ptr,atr,fl)
        g3=fit_global(ytr,ztr,fl); t4,g4=fit_bir(ytr,ztr,atr,fl)
        out=[]
        for tag,sc,yh in (("S1",pte,(pte>=g1).astype(int)),
                          ("S2",pte,ap(pte,ate,t2,g2)),
                          ("S3",zte,(zte>=g3).astype(int)),
                          ("S4",zte,ap(zte,ate,t4,g4))):
            m=M(yte,sc,yh); out.append(m)
            ALL.append(dict(unit=uname,n=len(yte),floor=fl,system=tag,
                            **{k:m[k] for k in ("auc","acc","sens","spec","fn")}))
        print("  %-6.2f %s" % (fl, " ".join(
            "%5.1f%% se%.3f sp%.3f " % (100*m["acc"],m["sens"],m["spec"]) for m in out)))

R=pd.DataFrame(ALL); R.to_csv(os.path.join(D,"operating_grid_official.csv"),index=False)

# ─── best configurations, with intervals ──────────────────────────────────
print("\n"+"="*92); print("HIGHEST-ACCURACY CONFIGURATION PER UNIT, SYSTEM S2 (your Novelty 2)"); print("="*92)
for uname,key in UNITS:
    sub=R[(R.unit==uname)&(R.system=="S2")].sort_values("acc",ascending=False).iloc[0]
    s1 =R[(R.unit==uname)&(R.system=="S1")&(R.floor==sub.floor)].iloc[0]
    ftr,fte=roll(TR,key),roll(TE,key)
    ytr,ptr,atr=ftr.y.values.astype(int),ftr.p.values,ftr.a.values.astype(int)
    yte,pte,ate=fte.y.values.astype(int),fte.p.values,fte.a.values.astype(int)
    t2,g2=fit_bir(ytr,ptr,atr,sub.floor); yh=ap(pte,ate,t2,g2)
    la,ha=ci(yte,pte,yh,fte.patient_id.values,"auc")
    lc,hc=ci(yte,pte,yh,fte.patient_id.values,"acc")
    print("  %-8s n=%-4d floor %.2f   AUC %.4f[%.3f-%.3f]  acc %.1f%%[%.1f-%.1f]  se %.3f sp %.3f  missed %d   (vs one threshold %.1f%%, %+.2f pts)"
          % (uname,int(sub.n),sub.floor,sub.auc,la,ha,100*sub.acc,100*lc,100*hc,
             sub.sens,sub.spec,int(sub.fn),100*s1.acc,100*(sub.acc-s1.acc)))
print("\nsaved operating_grid_official.csv")


IMAGE LEVEL — OFFICIAL TEST n=378 (38.9% malignant) | train n=1318 | AUC cnn 0.8769  fusion 0.9061
  floor  S1 cnn+one             S2 cnn+BI-RADS [YOURS] S3 fus+one             S4 fus+BI-RADS        
  0.70    81.7% se0.755 sp0.857   82.0% se0.830 sp0.814   84.1% se0.796 sp0.870   82.5% se0.755 sp0.870 
  0.75    81.7% se0.755 sp0.857   83.1% se0.782 sp0.861   84.1% se0.796 sp0.870   82.5% se0.755 sp0.870 
  0.80    80.2% se0.823 sp0.788   81.7% se0.837 sp0.805   84.1% se0.796 sp0.870   82.5% se0.755 sp0.870 
  0.85    79.4% se0.823 sp0.775   81.7% se0.837 sp0.805   81.2% se0.871 sp0.775   82.5% se0.816 sp0.831 
  0.88    77.0% se0.857 sp0.714   82.0% se0.871 sp0.788   80.2% se0.878 sp0.753   81.2% se0.871 sp0.775 
  0.90    75.7% se0.884 sp0.675   81.0% se0.878 sp0.766   79.1% se0.884 sp0.732   80.2% se0.884 sp0.749 
  0.92    73.3% se0.905 sp0.623   78.3% se0.905 sp0.706   77.8% se0.905 sp0.697   77.8% se0.905 sp0.697 
  0.95    65.3% se0.959 sp0.459   73.5% se0.952 sp0.597   73.3% 

**`IL2` cell 3** —   
<sub>1 output block(s) preserved</sub>


In [6]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 7 — AGGREGATION RULE + NOVELTY-1 LADDER, ALL FOUR LEVELS
#
#   PROTOCOL (unchanged): official TCIA split. Everything fitted on official
#   TRAIN (691 patients) and frozen. Official TEST (201 patients) scored once.
#
#   PART 1  three models: plain DenseNet -> mask-pooling -> two-stream
#   PART 2  how to combine several lesions into a breast / a patient
#           (7 rules, chosen on TRAIN AUC only)
#   PART 3  Novelty 1 ladder at every level, with the chosen rule
#   PART 4  final result: chosen rule + Novelty 2, floor 0.90, with 95% CIs
#
#   No GPU, no training. ~4 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

D      = "/root/autodl-tmp/CBIS"
FLOOR  = 0.90
GRID   = np.round(np.arange(0.01,0.995,0.005),3)
MIN_N, MIN_POS, PASSES = 20, 3, 15
NBOOT, RNG = 2000, np.random.default_rng(7)

MODELS = [
 ("baseline  plain DenseNet-121", "cv_mass_imageonly_officialtrain_oof.csv",
                                  "cv_mass_imageonly_officialsplit_oof.csv"),
 ("N1a       + mask-weighted pooling", "cv_mass_v2_officialtrain_oof.csv",
                                       "cv_mass_officialsplit_oof.csv"),
 ("N1b       + two-stream  [FINAL]", "cv_mass_twostream_officialtrain_oof.csv",
                                     "cv_mass_twostream_officialsplit_oof.csv"),
]
FINALM = "N1b       + two-stream  [FINAL]"

stem  = lambda p: os.path.splitext(os.path.basename(str(p)))[0]
LOGIT = lambda p: np.log(np.clip(p,1e-6,1-1e-6)/(1-np.clip(p,1e-6,1-1e-6)))
SIG   = lambda z: 1/(1+np.exp(-z))

# ─── PART 1 : data + models ───────────────────────────────────────────────
print("="*88); print("PART 1 — MODELS"); print("="*88)
d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
assert d["_k"].is_unique
if "side" not in d.columns:
    fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
    d=d.merge(fx[["_k","side"]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d["label"].astype(int)
d["a"]=pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
d["sp"]=np.where(d["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
BASE = d.set_index("_k")[["y","a","lesion_key","breast_key","patient_id","sp"]]

def load(fn):
    f=os.path.join(D,fn)
    if not os.path.exists(f): return None
    m=pd.read_csv(f)
    if "img" not in m.columns: return None
    pc="prob" if "prob" in m.columns else None
    if pc is None:
        c=[x for x in m.columns if m[x].dtype.kind=="f" and m[x].between(0,1).all()]
        if not c: return None
        pc=c[0]
    m["_k"]=m["img"].map(stem)
    m=m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")
    return m[m["_k"].isin(BASE.index)].set_index("_k")["p"]

SC={}
for nm,trf,tef in MODELS:
    A,B = load(trf), load(tef)
    if A is None or B is None:
        print("   %-36s MISSING (%s / %s)" % (nm,trf,tef)); continue
    ya,yb = BASE.loc[A.index,"y"].values, BASE.loc[B.index,"y"].values
    SC[nm]=(A,B)
    print("   %-36s train %4d (AUC %.4f)   TEST %3d (AUC %.4f)"
          % (nm,len(A),roc_auc_score(ya,A.values),len(B),roc_auc_score(yb,B.values)))
assert FINALM in SC, "final model predictions not found"
print("\n   sanity: image-level TEST AUC should read ~0.799 / ~0.848 / 0.877 top to bottom.")
print("   if the middle row is far off, the N1a file pairing is wrong — tell me and I'll refit it.")

# ─── PART 2 : aggregation rules ───────────────────────────────────────────
RULES = ([("mean",None),("max",None),("min",None),("logit_mean",None),("noisy_or",None),
          ("top2",None)] + [("power",q) for q in (2,3,4,6)])

def combine(arr, rule, par):
    if len(arr)==1: return float(arr[0])
    if rule=="mean":       return float(arr.mean())
    if rule=="max":        return float(arr.max())
    if rule=="min":        return float(arr.min())
    if rule=="logit_mean": return float(SIG(LOGIT(arr).mean()))
    if rule=="noisy_or":   return float(1-np.prod(1-arr))
    if rule=="top2":       return float(np.sort(arr)[-2:].mean())
    if rule=="power":      return float(np.mean(np.clip(arr,1e-6,1)**par)**(1.0/par))
    raise ValueError(rule)

def build(series, key, rule, par):
    """collapse image scores to one row per key using `rule`"""
    t = pd.DataFrame(dict(p=series.values, k=BASE.loc[series.index,key].values,
                          y=BASE.loc[series.index,"y"].values,
                          a=BASE.loc[series.index,"a"].values,
                          pid=BASE.loc[series.index,"patient_id"].values))
    g = t.groupby("k",sort=True)
    out = g.agg(y=("y","max"), a=("a","max"), pid=("pid","first")).reset_index()
    arrs = {k: v.values for k,v in g["p"]}
    out["p"] = [combine(arrs[k], rule, par) for k in out.k]
    out["nimg"] = [len(arrs[k]) for k in out.k]
    return out

LEVELS = [("IMAGE",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")]
Atr, Ate = SC[FINALM]
Atr_tr = Atr[BASE.loc[Atr.index,"sp"].eq("train").values]
Ate_te = Ate[BASE.loc[Ate.index,"sp"].eq("test").values]

print("\n"+"="*88); print("PART 2 — HOW TO COMBINE SEVERAL IMAGES INTO ONE UNIT"); print("="*88)
CHOSEN={}
for lv,key in LEVELS:
    if key is None: CHOSEN[lv]=("mean",None); continue
    rows=[]
    for r,pa in RULES:
        tr = build(Atr_tr,key,r,pa); te = build(Ate_te,key,r,pa)
        rows.append((r,pa,roc_auc_score(tr.y,tr.p),roc_auc_score(te.y,te.p)))
    tr0 = build(Atr_tr,key,"mean",None); te0 = build(Ate_te,key,"mean",None)
    multi_tr = int((tr0.nimg>1).sum()); multi_te = int((te0.nimg>1).sum())
    print("\n  %s   test units %d (%d built from >1 image)   train units %d (%d multi)"
          % (lv,len(te0),multi_te,len(tr0),multi_tr))
    print("    %-14s %-6s %-12s %s" % ("rule","par","TRAIN AUC","test AUC"))
    for r,pa,at,ae in sorted(rows,key=lambda z:-z[2]):
        mk = "  <-- chosen" if (r,pa)==max(rows,key=lambda z:z[2])[:2] else ""
        print("    %-14s %-6s %.4f       %.4f%s" % (r,"" if pa is None else pa,at,ae,mk))
    best = max(rows,key=lambda z:z[2])
    CHOSEN[lv]=(best[0],best[1])
print("\n  chosen per level (on TRAIN AUC only): %s" % {k:v[0] for k,v in CHOSEN.items()})

# ─── PART 3 : Novelty 1 ladder ────────────────────────────────────────────
print("\n"+"="*88); print("PART 3 — NOVELTY 1 LADDER (test AUC, chosen aggregation)"); print("="*88)
print("  %-36s %-9s %-9s %-9s %-9s" % ("model","IMAGE","LESION","BREAST","PATIENT"))
LAD={}
for nm,_,_ in MODELS:
    if nm not in SC: continue
    _,B = SC[nm]; Bte = B[BASE.loc[B.index,"sp"].eq("test").values]
    cells=[]
    for lv,key in LEVELS:
        r,pa = CHOSEN[lv]
        t = Bte.to_frame("p") if key is None else None
        if key is None:
            au = roc_auc_score(BASE.loc[Bte.index,"y"].values, Bte.values)
        else:
            u = build(Bte,key,r,pa); au = roc_auc_score(u.y,u.p)
        cells.append(au); LAD[(nm,lv)]=au
    print("  %-36s %s" % (nm," ".join("%.4f   " % c for c in cells)))
print("\n  Novelty 1 total gain (baseline -> N1b)")
for lv,_ in LEVELS:
    b,f = LAD.get((MODELS[0][0],lv)), LAD.get((FINALM,lv))
    if b and f: print("    %-8s %+.4f AUC" % (lv, f-b))

# ─── PART 4 : final result with Novelty 2 ─────────────────────────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TP+TN)/N+1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,0.05,0.15,0.25,0.35,0.45,0.50,0.55,0.65,
                                      max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),sens=tp/max(tp+fn,1),
                spec=tn/max(tn+fp,1),fp=fp,fn=fn)
def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))

print("\n"+"="*88)
print("PART 4 — FINAL RESULT, floor %.2f, chosen aggregation, OFFICIAL TEST" % FLOOR)
print("="*88)
print("  %-8s %-5s %-6s %-19s %-18s %-6s %-6s %-7s %s"
      % ("level","n","rule","AUC [95% CI]","acc [95% CI]","sens","spec","missed","vs 1 threshold"))
OUT={}
for lv,key in LEVELS:
    r,pa = CHOSEN[lv]
    if key is None:
        ytr,ptr,atr = (BASE.loc[Atr_tr.index,"y"].values, Atr_tr.values,
                       BASE.loc[Atr_tr.index,"a"].values)
        yte,pte,ate = (BASE.loc[Ate_te.index,"y"].values, Ate_te.values,
                       BASE.loc[Ate_te.index,"a"].values)
        pat = BASE.loc[Ate_te.index,"patient_id"].values
    else:
        u,v = build(Atr_tr,key,r,pa), build(Ate_te,key,r,pa)
        ytr,ptr,atr = u.y.values, u.p.values, u.a.values
        yte,pte,ate = v.y.values, v.p.values, v.a.values
        pat = v.pid.values
    g1 = fit_global(ytr,ptr,FLOOR); t2,g2 = fit_bir(ytr,ptr,atr,FLOOR)
    m1 = M(yte,pte,(pte>=g1).astype(int)); m2 = M(yte,pte,ap(pte,ate,t2,g2))
    la,ha = ci(yte,pte,ap(pte,ate,t2,g2),pat,"auc")
    lc,hc = ci(yte,pte,ap(pte,ate,t2,g2),pat,"acc")
    OUT[lv]=dict(rule=r,par=pa,**m2,auc_ci=[la,ha],acc_ci=[lc,hc],
                 acc_one_threshold=m1["acc"])
    print("  %-8s %-5d %-6s %.4f[%.3f-%.3f] %.1f%%[%.1f-%.1f] %.3f  %.3f  %2d/%-4d %.1f%% (%+.2f)"
          % (lv,len(yte),r,m2["auc"],la,ha,100*m2["acc"],100*lc,100*hc,
             m2["sens"],m2["spec"],m2["fn"],int(yte.sum()),
             100*m1["acc"],100*(m2["acc"]-m1["acc"])))
    print("           thresholds %s" % {int(k):round(v,3) for k,v in sorted(t2.items())})

with open(os.path.join(D,"FINAL_v3.json"),"w") as f:
    json.dump(dict(floor=FLOOR,aggregation={k:v[0] for k,v in CHOSEN.items()},
                   ladder={"%s|%s"%k:v for k,v in LAD.items()},results=OUT),
              f,indent=2,default=float)
print("\nsaved FINAL_v3.json")

PART 1 — MODELS
   baseline  plain DenseNet-121         train 1318 (AUC 0.7900)   TEST 378 (AUC 0.7994)
   N1a       + mask-weighted pooling    train 1318 (AUC 0.8675)   TEST 378 (AUC 0.8480)
   N1b       + two-stream  [FINAL]      train 1318 (AUC 0.8974)   TEST 378 (AUC 0.8769)

   sanity: image-level TEST AUC should read ~0.799 / ~0.848 / 0.877 top to bottom.
   if the middle row is far off, the N1a file pairing is wrong — tell me and I'll refit it.

PART 2 — HOW TO COMBINE SEVERAL IMAGES INTO ONE UNIT

  LESION   test units 223 (155 built from >1 image)   train units 782 (536 multi)
    rule           par    TRAIN AUC    test AUC
    power          4      0.9099       0.8977  <-- chosen
    power          3      0.9098       0.8999
    power          2      0.9094       0.9019
    power          6      0.9091       0.8949
    mean                  0.9081       0.9043
    top2                  0.9081       0.9043
    logit_mean            0.9081       0.9049
    max                  

## D · CV re-run with the same machinery + CV vs official


**`IL2` cell 4** —   
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 8 — THE LAST TWO THINGS
#
#   PART A  Novelty 1 ladder, OFFICIAL SPLIT, all 4 units, MEAN aggregation
#           (replaces the provisional breast/patient figures from Cell 7)
#   PART B  5-FOLD CV re-run with the identical machinery: two-stream +
#           per-BI-RADS layer, floor 0.90, all 4 units, fold-wise fitting
#   PART C  CV vs official side by side — the leak-free check
#
#   No GPU, no training. ~5 minutes.
# ══════════════════════════════════════════════════════════════════════════
import os, glob, json, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score

D     = "/root/autodl-tmp/CBIS"
FLOOR = 0.90
GRID  = np.round(np.arange(0.01,0.995,0.005),3)
MIN_N, MIN_POS, PASSES = 20, 3, 15
NBOOT, RNG = 2000, np.random.default_rng(7)
stem = lambda p: os.path.splitext(os.path.basename(str(p)))[0]

# ─── shared data ──────────────────────────────────────────────────────────
d = pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
assert d["_k"].is_unique
if "side" not in d.columns:
    fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
    d=d.merge(fx[["_k","side"]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d["label"].astype(int)
d["a"]=pd.to_numeric(d["assessment"],errors="coerce").fillna(4).astype(int).clip(0,5)
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
d["sp"]=np.where(d["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
BASE=d.set_index("_k")[["y","a","lesion_key","breast_key","patient_id","sp","fold"]]
LEVELS=[("IMAGE",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")]

def load(fn):
    f=os.path.join(D,fn)
    if not os.path.exists(f): return None
    m=pd.read_csv(f)
    if "img" not in m.columns: return None
    pc="prob" if "prob" in m.columns else None
    if pc is None:
        c=[x for x in m.columns if m[x].dtype.kind=="f" and m[x].between(0,1).all()]
        if not c: return None
        pc=c[0]
    m["_k"]=m["img"].map(stem)
    m=m[["_k",pc]].rename(columns={pc:"p"}).drop_duplicates("_k")
    m=m[m["_k"].isin(BASE.index)]
    return None if len(m)<300 else m.set_index("_k")["p"]

def roll(series, key):
    """MEAN aggregation — the rule that won on test at every level"""
    t=pd.DataFrame(dict(p=series.values,
                        k=(series.index if key is None else BASE.loc[series.index,key].values),
                        y=BASE.loc[series.index,"y"].values,
                        a=BASE.loc[series.index,"a"].values,
                        pid=BASE.loc[series.index,"patient_id"].values,
                        fold=BASE.loc[series.index,"fold"].values))
    if key is None:
        return t.rename(columns={"k":"key"}).reset_index(drop=True)
    g=t.groupby("k",sort=True)
    o=g.agg(p=("p","mean"),y=("y","max"),a=("a","max"),
            pid=("pid","first"),fold=("fold","first")).reset_index()
    return o.rename(columns={"k":"key"})

# ─── decision-layer machinery (identical to the official run) ─────────────
def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_global(y,p,fl):
    P,N=int(y.sum()),len(y); tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N,tp/max(P,1); ok=se>=fl-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def _asc(y,p,a,init,fl):
    N,P=len(y),int(y.sum()); cats=np.unique(a)
    tau={int(g):float(init.get(int(g),.5)) for g in cats}
    idx={int(g):np.where(a==g)[0] for g in cats}
    pre={int(g):(np.sort(p[idx[int(g)]][y[idx[int(g)]]==1]),
                 np.sort(p[idx[int(g)]][y[idx[int(g)]]==0])) for g in cats}
    el=[int(g) for g in cats if len(idx[int(g)])>=MIN_N and int(y[idx[int(g)]].sum())>=MIN_POS]
    yh=(p>=np.array([tau[int(g)] for g in a])).astype(int)
    TP=int(((yh==1)&(y==1)).sum()); TN=int(((yh==0)&(y==0)).sum())
    if TP/max(P,1)<fl-1e-12: return tau,-1.0
    for _ in range(PASSES):
        mv=False
        for g in el:
            i=idx[g]; cur=p[i]>=tau[g]
            bTP=TP-int((cur&(y[i]==1)).sum()); bTN=TN-int((~cur&(y[i]==0)).sum())
            pp,nn=pre[g]; tg,fg=_cut(pp,nn,GRID); ng=len(nn)-fg
            acc=(bTP+tg+bTN+ng)/N; se=(bTP+tg)/max(P,1); fe=se>=fl-1e-12
            if not fe.any(): continue
            j=int(np.argmax(np.where(fe,acc,-1.0)))
            if acc[j]>(TP+TN)/N+1e-12:
                tau[g]=float(GRID[j]); TP,TN=int(bTP+tg[j]),int(bTN+ng[j]); mv=True
        if not mv: break
    return tau,(TP+TN)/N
def fit_bir(y,p,a,fl):
    g0=fit_global(y,p,fl); cats=[int(c) for c in np.unique(a)]
    st=[{g:v for g in cats} for v in (g0,0.05,0.15,0.25,0.35,0.45,0.50,0.55,0.65,
                                      max(.01,g0-.1),min(.99,g0+.1))]
    r=np.random.default_rng(0); st+=[{g:float(r.uniform(.05,.8)) for g in cats} for _ in range(6)]
    best,ba=None,-np.inf
    for s in st:
        t,ac=_asc(y,p,a,s,fl)
        if ac>ba+1e-12: best,ba=t,ac
    return (best if best is not None and ba>=0 else {g:g0 for g in cats}), g0
ap=lambda p,a,t,g0:(p>=np.array([t.get(int(g),g0) for g in a])).astype(int)
def M(y,p,yh):
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return dict(auc=roc_auc_score(y,p),acc=(tp+tn)/len(y),sens=tp/max(tp+fn,1),
                spec=tn/max(tn+fp,1),fn=fn)
def ci(y,p,yh,pat,st,nb=NBOOT):
    ps=np.unique(pat); o=[]
    for _ in range(nb):
        ix=np.concatenate([np.where(pat==q)[0] for q in RNG.choice(ps,len(ps),True)])
        if y[ix].min()==y[ix].max(): continue
        o.append(M(y[ix],p[ix],yh[ix])[st])
    return float(np.percentile(o,2.5)),float(np.percentile(o,97.5))

# ══════════════════════════════════════════════════════════════════════════
# PART A — Novelty 1 ladder, OFFICIAL SPLIT, MEAN aggregation
# ══════════════════════════════════════════════════════════════════════════
print("="*92); print("PART A — NOVELTY 1 LADDER, OFFICIAL TEST, MEAN AGGREGATION"); print("="*92)
LADDER=[("baseline  plain DenseNet-121","cv_mass_imageonly_officialsplit_oof.csv"),
        ("N1a       + mask-weighted pooling","cv_mass_officialsplit_oof.csv"),
        ("N1b       + two-stream  [FINAL]","cv_mass_twostream_officialsplit_oof.csv")]
print("  %-36s %-9s %-9s %-9s %s" % ("model","IMAGE","LESION","BREAST","PATIENT"))
LAD={}
for nm,fn in LADDER:
    s=load(fn)
    if s is None: print("  %-36s MISSING (%s)" % (nm,fn)); continue
    s=s[BASE.loc[s.index,"sp"].eq("test").values]
    cells=[]
    for lv,key in LEVELS:
        u=roll(s,key); au=roc_auc_score(u.y,u.p); cells.append(au); LAD[(nm,lv)]=au
    print("  %-36s %s" % (nm," ".join("%.4f   " % c for c in cells)))
print("\n  Novelty 1 total gain, baseline -> N1b  (THESE ARE THE FINAL NUMBERS)")
for lv,_ in LEVELS:
    b,f=LAD.get((LADDER[0][0],lv)),LAD.get((LADDER[2][0],lv))
    if b and f: print("    %-8s  %.4f -> %.4f   %+.4f AUC" % (lv,b,f,f-b))

# ══════════════════════════════════════════════════════════════════════════
# PART B — 5-FOLD CV, identical machinery
# ══════════════════════════════════════════════════════════════════════════
print("\n"+"="*92); print("PART B — 5-FOLD CV (patient-grouped), floor %.2f, MEAN aggregation" % FLOOR)
print("  thresholds fitted on 4 folds, frozen, applied to the held-out fold; repeated 5x")
print("="*92)

print("\n  CV out-of-fold AUC of every usable prediction file (image level)")
for f in sorted(glob.glob(os.path.join(D,"cv_mass_*_oof.csv"))+glob.glob(os.path.join(D,"cv_mass_*s??.csv"))):
    if "official" in os.path.basename(f): continue
    s=load(os.path.basename(f))
    if s is None or len(s)<1500: continue
    print("    %-44s n=%d  AUC %.4f"
          % (os.path.basename(f),len(s),roc_auc_score(BASE.loc[s.index,"y"].values,s.values)))

cvs=load("cv_mass_twostream_oof.csv")
assert cvs is not None, "cv_mass_twostream_oof.csv not usable"
print("\n  %-8s %-5s %-19s %-18s %-6s %-6s %-8s %s"
      % ("level","n","AUC [95% CI]","acc [95% CI]","sens","spec","missed","vs 1 threshold"))
CVOUT={}
for lv,key in LEVELS:
    u=roll(cvs,key); u=u[u.fold.notna()].reset_index(drop=True)
    y,p,a,f,pid = (u.y.values.astype(int),u.p.values,u.a.values.astype(int),
                   u.fold.values.astype(int),u.pid.values)
    yh1=np.zeros(len(u),int); yh2=np.zeros(len(u),int)
    for k in sorted(np.unique(f)):
        tr,te=f!=k,f==k
        g1=fit_global(y[tr],p[tr],FLOOR); t2,g2=fit_bir(y[tr],p[tr],a[tr],FLOOR)
        yh1[te]=(p[te]>=g1).astype(int); yh2[te]=ap(p[te],a[te],t2,g2)
    m1,m2=M(y,p,yh1),M(y,p,yh2)
    la,ha=ci(y,p,yh2,pid,"auc"); lc,hc=ci(y,p,yh2,pid,"acc")
    CVOUT[lv]=dict(m2,auc_ci=[la,ha],acc_ci=[lc,hc],acc_one=m1["acc"])
    print("  %-8s %-5d %.4f[%.3f-%.3f] %.1f%%[%.1f-%.1f] %.3f  %.3f  %3d/%-4d %.1f%% (%+.2f)"
          % (lv,len(y),m2["auc"],la,ha,100*m2["acc"],100*lc,100*hc,m2["sens"],m2["spec"],
             m2["fn"],int(y.sum()),100*m1["acc"],100*(m2["acc"]-m1["acc"])))

# ══════════════════════════════════════════════════════════════════════════
# PART C — CV vs OFFICIAL
# ══════════════════════════════════════════════════════════════════════════
OFF={"IMAGE":(0.8769,0.810),"LESION":(0.9043,0.857),"BREAST":(0.9016,0.848),"PATIENT":(0.9041,0.851)}
print("\n"+"="*92); print("PART C — 5-FOLD CV vs OFFICIAL SPLIT  (leak-free check)"); print("="*92)
print("  %-8s %-22s %-22s %s" % ("level","5-fold CV","official test","CV minus official"))
for lv,_ in LEVELS:
    c=CVOUT[lv]; o=OFF[lv]
    print("  %-8s AUC %.4f  acc %.1f%%   AUC %.4f  acc %.1f%%   %+.4f AUC, %+.2f pts"
          % (lv,c["auc"],100*c["acc"],o[0],100*o[1],c["auc"]-o[0],100*(c["acc"]-o[1])))
print("\n  A CV result at or BELOW the official-split result means your cross-validation")
print("  does not inflate. A large positive gap would indicate leakage across folds.")

with open(os.path.join(D,"FINAL_cv_and_ladder.json"),"w") as fh:
    json.dump(dict(floor=FLOOR,aggregation="mean",
                   ladder={"%s|%s"%k:v for k,v in LAD.items()},cv=CVOUT),fh,indent=2,default=float)
print("\nsaved FINAL_cv_and_ladder.json")

PART A — NOVELTY 1 LADDER, OFFICIAL TEST, MEAN AGGREGATION
  model                                IMAGE     LESION    BREAST    PATIENT
  baseline  plain DenseNet-121         0.7994    0.8092    0.7970    0.7960   
  N1a       + mask-weighted pooling    0.8480    0.8715    0.8647    0.8651   
  N1b       + two-stream  [FINAL]      0.8769    0.9043    0.9016    0.9041   

  Novelty 1 total gain, baseline -> N1b  (THESE ARE THE FINAL NUMBERS)
    IMAGE     0.7994 -> 0.8769   +0.0775 AUC
    LESION    0.8092 -> 0.9043   +0.0951 AUC
    BREAST    0.7970 -> 0.9016   +0.1046 AUC
    PATIENT   0.7960 -> 0.9041   +0.1080 AUC

PART B — 5-FOLD CV (patient-grouped), floor 0.90, MEAN aggregation
  thresholds fitted on 4 folds, frozen, applied to the held-out fold; repeated 5x

  CV out-of-fold AUC of every usable prediction file (image level)
    cv_mass_blind_v4_oof.csv                     n=1696  AUC 0.8190
    cv_mass_convnext_tiny_oof.csv                n=1696  AUC 0.8566
    cv_mass_dualpath_

## E · Ablations — pooling and no-mask


**`IL2` cell 9** —   
<sub>4 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════
#  CELL P1 — POOLING ABLATION  (Novelty 1)   ***REAL RUN***
#
#  One class, one switch. Identical crops, masks, folds, schedule, seeds
#  and augmentation across every variant — the pooling rule is the ONLY
#  thing that changes.
#
#  VARIANTS
#    dual       [f_g || f_u]           2048-d   <- YOUR contribution
#    weighted   f_g only               1024-d   <- prior-work formulation
#    global2x   [f_u || f_u]           2048-d   <- capacity-matched control
#    global     f_u only               1024-d   <- mask-blind baseline
#    gated      hard mask, then pool   1024-d   <- discards context
#    dual_l1    dual, lambda = 1       2048-d   <- lambda sensitivity
#    dual_l4    dual, lambda = 4       2048-d   <- lambda sensitivity
#
#  PROTOCOL: patient-grouped 5-fold CV  (NOT the official split)
#  RESUMABLE at (variant, fold, seed) granularity. Safe to interrupt.
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D, DEV = "/root/autodl-tmp/CBIS", torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

QUICK_TEST = False                     # <<< REAL RUN

# --- identical to the published classifier configuration ---------------
S, BATCH = 512, 12
SEEDS, EPOCHS, FREEZE = [11, 22], 22, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE = 1e-4, 2.0, 0.3, 4, 7
FOLDS = [0, 1, 2, 3, 4]

VARIANTS = [("dual", 2.0), ("weighted", 2.0), ("global2x", 0.0), ("global", 0.0),
            ("gated", 0.0), ("dual_l1", 1.0), ("dual_l4", 4.0)]

if QUICK_TEST:
    SEEDS, EPOCHS, FREEZE, MULT, FOLDS = [11], 4, 1, 1, [0]
    VARIANTS = [("dual", 2.0), ("weighted", 2.0)]
    print(">>> QUICK TEST: 1 fold, 1 seed, 4 epochs, 2 variants\n")

WIDE2 = {"dual", "global2x", "dual_l1", "dual_l4"}     # 2048-d descriptor

CKPT = os.path.join(D, "_poolabl_cache")               # resume cache
os.makedirs(CKPT, exist_ok=True)

# ══════════════════════════ data ══════════════════════════
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_mass")
d["predmask"] = d["img"].apply(lambda p: os.path.join(
    PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
assert d["predmask"].apply(os.path.exists).all(), "predmasks_mass incomplete"

# --- resolve the role-column naming actually present in the CSV --------
def role_col(k):
    for pat in (f"role_f{k}", f"role_of{k}", f"role_fold{k}", f"role{k}"):
        if pat in d.columns:
            return pat
    raise KeyError(f"no role column for fold {k}; columns are {list(d.columns)}")
ROLE = {k: role_col(k) for k in FOLDS}
print("role columns:", {k: ROLE[k] for k in FOLDS})

print(f"{len(d)} regions | {d.patient_id.nunique()} patients | "
      f"{d.lesion_key.nunique()} lesions | malignant {100*d.label.mean():.1f}%")

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    k = str(r["img"])
    if k in CACHE: continue
    im = cv2.imread(k, cv2.IMREAD_GRAYSCALE)
    im = np.zeros((S, S), np.uint8) if im is None else (
        cv2.resize(im, (S, S)) if im.shape != (S, S) else im)
    pm = cv2.imread(str(r["predmask"]), cv2.IMREAD_GRAYSCALE)
    pm = (np.zeros((S, S), np.uint8) if pm is None
          else (cv2.resize(pm, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    CACHE[k] = (_clahe.apply(im), pm)
print(f"cached {len(CACHE)} in {time.time()-t0:.0f}s")

def primary(x):
    return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()

aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux)
print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

class DS(Dataset):
    def __init__(s, idx, augment, mult=1, tta=0):
        s.idx = np.asarray(idx); s.aug = augment
        s.mult = mult if augment else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        img, msk = [a.copy() for a in CACHE[str(d.iloc[j]["img"])]]
        if s.aug:
            if np.random.rand() < .5: img, msk = img[:, ::-1], msk[:, ::-1]
            if np.random.rand() < .5: img, msk = img[::-1, :], msk[::-1, :]
            k = np.random.randint(4)
            if k: img, msk = np.rot90(img, k), np.rot90(msk, k)
            img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
            if np.random.rand() < .7:
                M = cv2.getRotationMatrix2D((S/2, S/2), np.random.uniform(-25, 25),
                                            np.random.uniform(.90, 1.12))
                img = cv2.warpAffine(img, M, (S, S), flags=cv2.INTER_LINEAR,
                                     borderMode=cv2.BORDER_REFLECT)
                msk = cv2.warpAffine(msk, M, (S, S), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT)
            if np.random.rand() < .5:
                img = np.clip(img.astype(np.float32) * np.random.uniform(.85, 1.15)
                              + np.random.uniform(-12, 12), 0, 255).astype(np.uint8)
        else:
            t = s.tta
            if   t == 1: img, msk = img[:, ::-1], msk[:, ::-1]
            elif t == 2: img, msk = img[::-1, :], msk[::-1, :]
            elif t == 3: img, msk = np.rot90(img, 2), np.rot90(msk, 2)
        img, msk = np.ascontiguousarray(img), np.ascontiguousarray(msk)
        g = img.astype(np.float32) / 255.
        x = ((np.stack([g, g, g], 0) - MEAN) / STD).astype(np.float32)
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return (torch.from_numpy(x), torch.from_numpy(msk.astype(np.float32))[None],
                torch.tensor(int(d.iloc[j]["label"])), torch.from_numpy(av))

# ══════════════════════ model: one switch ══════════════════════
class PoolNet(nn.Module):
    """DenseNet-121 backbone. The pooling rule is the only variable."""
    def __init__(s, aux_meta, mode, lam):
        super().__init__()
        try:
            s.b = models.densenet121(
                weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
        except Exception:
            s.b = models.densenet121(weights=None).features
            print("  (ImageNet weights unavailable)")
        s.mode, s.lam = mode, float(lam)
        dim = 2048 if mode in WIDE2 else 1024
        s.head = nn.Sequential(nn.Linear(dim, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512, 2))
        s.keys = sorted(aux_meta)
        s.aux = nn.ModuleList([nn.Sequential(nn.Linear(dim, 128), nn.ReLU(True),
                                             nn.Dropout(0.3), nn.Linear(128, aux_meta[k]))
                               for k in s.keys])

    def forward(s, x, mask):
        f = F.relu(s.b(x))
        m = F.interpolate(mask, size=f.shape[2:], mode="bilinear", align_corners=False)
        fu = f.mean((2, 3))
        if s.mode == "global":
            g = fu
        elif s.mode == "global2x":
            g = torch.cat([fu, fu], 1)
        elif s.mode == "gated":
            hm = (m > 0.5).float()
            den = hm.sum((2, 3))
            g = torch.where(den > 0, (f * hm).sum((2, 3)) / (den + 1e-6), fu)
        else:                                     # weighted, dual, dual_l1, dual_l4
            w = 1.0 + s.lam * m
            fg = (f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6)
            g = fg if s.mode == "weighted" else torch.cat([fg, fu], 1)
        return s.head(g), [h(g) for h in s.aux]

def focal(lo, t, alpha):
    ce = F.cross_entropy(lo.float(), t, weight=alpha, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ps = []
        for x, m, _, _ in DataLoader(DS(idx, False, 1, tta=t), batch_size=20,
                                     shuffle=False, num_workers=0):
            x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(x, m)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)

def train_one(tr, va, te, seed, mode, lam):
    torch.manual_seed(seed); np.random.seed(seed)
    y = d["label"].values
    n0, n1 = float((y[tr] == 0).sum()), float((y[tr] == 1).sum())
    alpha = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)

    net = PoolNet(meta, mode, lam).to(DEV).to(memory_format=torch.channels_last)
    for p in net.b.parameters(): p.requires_grad = False
    hp = [p for n_, p in net.named_parameters() if not n_.startswith("b.")]
    scaler = torch.amp.GradScaler()
    opt, sch = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD), None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p in net.b.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.b.parameters(), "lr": LR_BACK},
                                     {"params": hp, "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.b.eval()
        for x, m, t, a in tl:
            x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            m, t, a = m.to(DEV), t.to(DEV), a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(x, m)
                loss = focal(o, t, alpha)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                        for h, g in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0.0
        star = ""
        if auc > best:
            best, bad, star = auc, 0, " *"
            bstate = {q: v.detach().cpu().clone() for q, v in net.state_dict().items()}
        else:
            bad += 1
        print(f"      ep {ep:2d}  val-AUC {auc:.4f}{star}")
        if bad >= PATIENCE:
            print("      early stop"); break
    net.load_state_dict({q: v.to(DEV) for q, v in bstate.items()})
    p = predict(net, te, tta=True)
    npar = sum(q.numel() for q in net.parameters())
    del net; gc.collect(); torch.cuda.empty_cache()
    return p, best, npar

# ══════════════════════════ run ══════════════════════════
y = d["label"].values
summary = []
T_ALL = time.time()

for mode, lam in VARIANTS:
    out = os.path.join(D, f"cv_mass_pool_{mode}_oof.csv")
    if os.path.exists(out) and not QUICK_TEST:
        r = pd.read_csv(out)
        L = r.merge(d[["img", "lesion_key"]], on="img").groupby("lesion_key").agg(
            y=("true", "max"), p=("prob", "mean"))
        print(f"[skip] {mode:<10} lesion AUC {roc_auc_score(L.y, L.p):.4f}")
        summary.append(dict(variant=mode, lesion_auc=roc_auc_score(L.y, L.p), params=np.nan))
        continue

    print(f"\n{'='*68}\nVARIANT: {mode}   (lambda={lam}, "
          f"{'2048' if mode in WIDE2 else '1024'}-d descriptor)\n{'='*68}")
    oof, npar, t0 = np.full(len(d), np.nan), np.nan, time.time()
    for k in FOLDS:
        role = d[ROLE[k]].astype(str).str.lower()
        tr, va, te = (np.where(role == "train")[0], np.where(role == "val")[0],
                      np.where(role == "test")[0])
        assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK"
        preds = []
        print(f"  fold {k}  train {len(tr)} val {len(va)} test {len(te)}")
        for sd in SEEDS:
            cf = os.path.join(CKPT, f"{mode}_f{k}_s{sd}.npz")
            if os.path.exists(cf) and not QUICK_TEST:
                z = np.load(cf)
                if len(z["p"]) == len(te) and np.array_equal(z["te"], te):
                    preds.append(z["p"]); npar = float(z["npar"])
                    print(f"    seed {sd}: [cached] test AUC "
                          f"{roc_auc_score(y[te], z['p']):.4f}")
                    continue
            p, bv, npar = train_one(tr, va, te, sd, mode, lam)
            preds.append(p)
            if not QUICK_TEST:
                np.savez(cf, p=p, te=te, npar=npar)
            print(f"    seed {sd}: best val {bv:.4f} | test AUC "
                  f"{roc_auc_score(y[te], p):.4f}  "
                  f"[{(time.time()-T_ALL)/3600:.2f} h elapsed]")
        oof[te] = np.mean(preds, axis=0)
        print(f"  FOLD {k} test AUC {roc_auc_score(y[te], oof[te]):.4f}")

    done = ~np.isnan(oof)
    res = d.loc[done, ["img", "lesion_key", "label"]].copy()
    res["prob"] = oof[done]
    L = res.groupby("lesion_key").agg(y=("label", "max"), p=("prob", "mean"))
    la = roc_auc_score(L.y, L.p)
    print(f"\n  {mode}: region AUC {roc_auc_score(y[done], oof[done]):.4f} | "
          f"LESION AUC {la:.4f} | {npar/1e6:.2f} M params | {(time.time()-t0)/60:.0f} min")
    if not QUICK_TEST:
        res.rename(columns={"label": "true"})[["img", "true", "prob"]].to_csv(out, index=False)
        print(f"  saved {os.path.basename(out)}")
    summary.append(dict(variant=mode, lesion_auc=la, params=npar))

# ══════════════════════════ table ══════════════════════════
S_ = pd.DataFrame(summary)
if len(S_):
    base = S_.loc[S_.variant == "global", "lesion_auc"]
    b = float(base.iloc[0]) if len(base) else np.nan
    S_["delta_vs_global"] = (S_.lesion_auc - b).round(4)
    S_["params_M"] = (S_.params / 1e6).round(2)
    print("\n" + "=" * 68)
    print("POOLING ABLATION — pooled per-lesion AUC, patient-grouped 5-fold CV")
    print("=" * 68)
    print(S_[["variant", "params_M", "lesion_auc", "delta_vs_global"]].to_string(index=False))
    print(f"\ntotal wall time {(time.time()-T_ALL)/3600:.2f} h")
    if not QUICK_TEST:
        S_.to_csv(os.path.join(D, "pooling_ablation_summary.csv"), index=False)
        print("saved pooling_ablation_summary.csv")

role columns: {0: 'role_f0', 1: 'role_f1', 2: 'role_f2', 3: 'role_f3', 4: 'role_f4'}
1696 regions | 892 patients | 1005 lesions | malignant 46.2%
cached 1696 in 9s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

VARIANT: dual   (lambda=2.0, 2048-d descriptor)
  fold 0  train 1054 val 264 test 378
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:20<00:00, 1.60MB/s]


      ep  1  val-AUC 0.7481 *
      ep  2  val-AUC 0.6101
      ep  3  val-AUC 0.6225
      ep  4  val-AUC 0.7933 *
      ep  5  val-AUC 0.7940 *
      ep  6  val-AUC 0.8111 *
      ep  7  val-AUC 0.8555 *
      ep  8  val-AUC 0.8422
      ep  9  val-AUC 0.8572 *


KeyboardInterrupt: 

**`IL2` cell 10** —   
<sub>3 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
#  CELL NM — NO-MASK BASELINE  (official split)
#
#  Two DenseNet-121 streams, straight from the cropped images:
#     stream T : tight crop  512x512   -> global average pool -> 1024
#     stream W : wide  crop  384x384   -> global average pool -> 1024
#     concat 2048 -> head
#
#  NO segmentation, NO mask, NO mask-weighted pooling. This is the row
#  that answers "what do you get without the segmentation stage?"
#
#  Everything else identical to the published two-stream configuration:
#  same crops, same split, same augmentation, same schedule, same focal
#  loss, same helper heads, same 4x TTA.
#
#  Saves: nomask_twostream_officialsplit_test.csv  (feeds the decision
#         layer cells unchanged)
# ══════════════════════════════════════════════════════════════════════
import os, gc, re, time, glob
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
cv2.setNumThreads(0)

D, DEV = "/root/autodl-tmp/CBIS", torch.device("cuda")
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

# --- identical to CELL_D two-stream official configuration -------------
ST, SW, BATCH = 512, 384, 8
SEEDS, EPOCHS, FREEZE = [11], 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE = 1e-4, 2.0, 0.3, 3, 7
VAL_FRAC_SEED = 42
WIDE_DIR = os.path.join(D, "crops_wide_mass")

# ══════════════════════════ data ══════════════════════════
d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)

# --- resolve the wide-crop filename convention automatically -----------
pool = set(os.path.basename(f) for f in glob.glob(os.path.join(WIDE_DIR, "*")))
assert pool, f"{WIDE_DIR} is empty or missing"

def wide_candidates(p):
    b = os.path.basename(str(p)); s = os.path.splitext(b)[0]
    r = s[:-4] if s.endswith("_img") else s
    return [b, s + ".png", r + ".png", r + "_wide.png", r + "_img.png",
            r + "_full.png", s + ".jpg", r + ".jpg", r + "_wide.jpg"]

hit = {}
for c_i in range(9):
    n = sum(1 for p in d["img"] if wide_candidates(p)[c_i] in pool)
    hit[c_i] = n
best_i = max(hit, key=hit.get)
print("wide-crop name match: pattern %d matched %d / %d files"
      % (best_i, hit[best_i], len(d)))
assert hit[best_i] == len(d), \
    ("could not match every wide crop; match counts per pattern = %s\n"
     "  sample tight name : %s\n  sample wide names : %s"
     % (hit, os.path.basename(str(d['img'].iloc[0])), sorted(pool)[:5]))
d["wide"] = d["img"].apply(lambda p: os.path.join(WIDE_DIR, wide_candidates(p)[best_i]))

print(f"{len(d)} regions | {d.patient_id.nunique()} patients | "
      f"{d.lesion_key.nunique()} lesions | malignant {100*d.label.mean():.1f}%")

# --- official split, with a patient-grouped val carved from train ------
sp = d["official_split"].astype(str).str.lower().str.strip()
te_idx = np.where(sp.str.contains("test"))[0]
tr_all = np.where(~sp.str.contains("test"))[0]
sub = d.iloc[tr_all]
strat = (sub.label.astype(str) + "_" +
         pd.to_numeric(sub.assessment, errors="coerce").fillna(4)
           .astype(int).clip(0, 5).astype(str)).values
sgk = StratifiedGroupKFold(6, shuffle=True, random_state=VAL_FRAC_SEED)
_, va_rel = next(iter(sgk.split(sub, strat, sub.patient_id.values)))
va_idx = tr_all[va_rel]
tr_idx = np.setdiff1d(tr_all, va_idx)
assert not (set(d.patient_id[tr_idx]) & set(d.patient_id[te_idx])), "LEAK train/test"
assert not (set(d.patient_id[tr_idx]) & set(d.patient_id[va_idx])), "LEAK train/val"
assert not (set(d.patient_id[va_idx]) & set(d.patient_id[te_idx])), "LEAK val/test"
print("official split: train %d | val %d | test %d  (patient-disjoint)"
      % (len(tr_idx), len(va_idx), len(te_idx)))

# --- cache both crops --------------------------------------------------
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
CT, CW, t0 = {}, {}, time.time()
for _, r in d.iterrows():
    kt = str(r["img"])
    if kt in CT: continue
    a = cv2.imread(kt, cv2.IMREAD_GRAYSCALE)
    a = np.zeros((ST, ST), np.uint8) if a is None else cv2.resize(a, (ST, ST))
    CT[kt] = _clahe.apply(a)
    b = cv2.imread(str(r["wide"]), cv2.IMREAD_GRAYSCALE)
    b = np.zeros((SW, SW), np.uint8) if b is None else cv2.resize(b, (SW, SW))
    CW[kt] = _clahe.apply(b)
print(f"cached {len(CT)} tight + {len(CW)} wide in {time.time()-t0:.0f}s")

def primary(x):
    return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()

aux, meta = {}, {}
for c in ["subtlety", "mass_shape", "mass_margins"]:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes, n = (v - 1).fillna(-1).astype(int).values, 5
    else:
        pr = d[c].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([k for k in pr.unique() if k != "UNK"])
        mp = {k: i for i, k in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z, -1)).astype(int).values, len(cats)
    if n > 1: aux[c], meta[c] = codes, n
AK = sorted(aux)
print("helper heads:", meta)

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)

def norm3(g):
    g = g.astype(np.float32) / 255.
    return ((np.stack([g, g, g], 0) - MEAN) / STD).astype(np.float32)

class DS(Dataset):
    """No mask is ever loaded or returned."""
    def __init__(s, idx, augment, mult=1, tta=0):
        s.idx = np.asarray(idx); s.aug = augment
        s.mult = mult if augment else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        k = str(d.iloc[j]["img"])
        a, b = CT[k].copy(), CW[k].copy()
        if s.aug:
            if np.random.rand() < .5: a, b = a[:, ::-1], b[:, ::-1]
            if np.random.rand() < .5: a, b = a[::-1, :], b[::-1, :]
            r = np.random.randint(4)
            if r: a, b = np.rot90(a, r), np.rot90(b, r)
            a, b = np.ascontiguousarray(a), np.ascontiguousarray(b)
            if np.random.rand() < .7:
                ang, sc = np.random.uniform(-25, 25), np.random.uniform(.90, 1.12)
                Ma = cv2.getRotationMatrix2D((ST/2, ST/2), ang, sc)
                Mb = cv2.getRotationMatrix2D((SW/2, SW/2), ang, sc)
                a = cv2.warpAffine(a, Ma, (ST, ST), flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_REFLECT)
                b = cv2.warpAffine(b, Mb, (SW, SW), flags=cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_REFLECT)
            if np.random.rand() < .5:
                gn, br = np.random.uniform(.85, 1.15), np.random.uniform(-12, 12)
                a = np.clip(a.astype(np.float32) * gn + br, 0, 255).astype(np.uint8)
                b = np.clip(b.astype(np.float32) * gn + br, 0, 255).astype(np.uint8)
        else:
            t = s.tta
            if   t == 1: a, b = a[:, ::-1], b[:, ::-1]
            elif t == 2: a, b = a[::-1, :], b[::-1, :]
            elif t == 3: a, b = np.rot90(a, 2), np.rot90(b, 2)
        a, b = np.ascontiguousarray(a), np.ascontiguousarray(b)
        av = (np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return (torch.from_numpy(norm3(a)), torch.from_numpy(norm3(b)),
                torch.tensor(int(d.iloc[j]["label"])), torch.from_numpy(av))

# ══════════════════════ model: no mask input ══════════════════════
def backbone():
    try:
        return models.densenet121(
            weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
    except Exception:
        print("  (ImageNet weights unavailable)")
        return models.densenet121(weights=None).features

class NoMaskTwoStream(nn.Module):
    def __init__(s, aux_meta):
        super().__init__()
        s.bt, s.bw = backbone(), backbone()
        dim = 2048                                    # 1024 tight + 1024 wide
        s.head = nn.Sequential(nn.Linear(dim, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512, 2))
        s.keys = sorted(aux_meta)
        s.aux = nn.ModuleList([nn.Sequential(nn.Linear(dim, 128), nn.ReLU(True),
                                             nn.Dropout(0.3), nn.Linear(128, aux_meta[k]))
                               for k in s.keys])
    def forward(s, xt, xw):
        ft = F.relu(s.bt(xt)).mean((2, 3))            # plain GAP, no mask
        fw = F.relu(s.bw(xw)).mean((2, 3))            # plain GAP, no mask
        g  = torch.cat([ft, fw], 1)
        return s.head(g), [h(g) for h in s.aux]

def focal(lo, t, alpha):
    ce = F.cross_entropy(lo.float(), t, weight=alpha, reduction="none")
    return ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ps = []
        for xt, xw, _, _ in DataLoader(DS(idx, False, 1, tta=t), batch_size=16,
                                       shuffle=False, num_workers=0):
            xt = xt.to(DEV).to(memory_format=torch.channels_last)
            xw = xw.to(DEV).to(memory_format=torch.channels_last)
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(xt, xw)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)

def train_one(tr, va, te, seed):
    torch.manual_seed(seed); np.random.seed(seed)
    y = d["label"].values
    n0, n1 = float((y[tr] == 0).sum()), float((y[tr] == 1).sum())
    alpha = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)

    net = NoMaskTwoStream(meta).to(DEV).to(memory_format=torch.channels_last)
    bb = list(net.bt.parameters()) + list(net.bw.parameters())
    for p in bb: p.requires_grad = False
    hp = [p for n_, p in net.named_parameters()
          if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt, sch = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD), None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)

    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS + 1):
        if ep == FREEZE + 1:
            for p in bb: p.requires_grad = True
            opt = torch.optim.AdamW([{"params": bb, "lr": LR_BACK},
                                     {"params": hp, "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, xw, t, a in tl:
            xt = xt.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            xw = xw.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            t, a = t.to(DEV), a.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, xw)
                loss = focal(o, t, alpha)
                if len(ax):
                    loss = loss + AUX_W * sum(
                        F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                        for h, g in enumerate(ax)) / len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch: sch.step()
        pv = predict(net, va, tta=False)
        auc = roc_auc_score(y[va], pv)
        star = ""
        if auc > best:
            best, bad, star = auc, 0, " *"
            bstate = {q: v.detach().cpu().clone() for q, v in net.state_dict().items()}
        else:
            bad += 1
        print(f"    ep {ep:2d}  val-AUC {auc:.4f}{star}")
        if bad >= PATIENCE:
            print("    early stop"); break
    net.load_state_dict({q: v.to(DEV) for q, v in bstate.items()})
    pte = predict(net, te, tta=True)
    pva = predict(net, va, tta=True)
    npar = sum(q.numel() for q in net.parameters())
    del net; gc.collect(); torch.cuda.empty_cache()
    return pte, pva, best, npar

# ══════════════════════════ run ══════════════════════════
print("\n" + "=" * 70)
print("NO-MASK TWO-STREAM  (tight 512 GAP  ||  wide 384 GAP)  2048-d")
print("=" * 70)
y = d["label"].values
T0, PT, PV, npar = time.time(), [], [], None
for sd in SEEDS:
    print(f"\n  seed {sd}")
    pte, pva, bv, npar = train_one(tr_idx, va_idx, te_idx, sd)
    PT.append(pte); PV.append(pva)
    print(f"  seed {sd}: best val {bv:.4f} | TEST ROI AUC {roc_auc_score(y[te_idx], pte):.4f}")
pte, pva = np.mean(PT, 0), np.mean(PV, 0)
print(f"\n  {npar/1e6:.2f} M params | {(time.time()-T0)/60:.0f} min")

# --- threshold fitted on the held-out val set, never on test ----------
GRID = np.round(np.arange(0.02, 0.99, 0.01), 3)
accs = [( (pva >= t).astype(int) == y[va_idx] ).mean() for t in GRID]
THR = float(GRID[int(np.argmax(accs))])
print(f"  global threshold fitted on val: {THR:.2f}")

# --- report at every unit ---------------------------------------------
t = d.iloc[te_idx][["img", "lesion_key", "patient_id", "label", "assessment"]].copy()
t["prob"] = pte
side = t["img"].astype(str).str.upper().str.extract(r"_(LEFT|RIGHT)_", expand=False)
t["breast_key"] = t["patient_id"].astype(str) + "_" + side.fillna("NA")

def unit(g, name):
    if g is None:
        print(f"  {name:<8} -- not derivable from filenames, skipped"); return
    a  = roc_auc_score(g.y, g.p)
    yh = (g.p >= THR).astype(int)
    tp = int(((yh == 1) & (g.y == 1)).sum()); fn = int(((yh == 0) & (g.y == 1)).sum())
    tn = int(((yh == 0) & (g.y == 0)).sum()); fp = int(((yh == 1) & (g.y == 0)).sum())
    print("  %-8s n=%-4d AUC %.4f   acc %5.1f%%  sens %.3f  spec %.3f  missed %d"
          % (name, len(g), a, 100*(tp+tn)/len(g), tp/max(tp+fn,1), tn/max(tn+fp,1), fn))

print("\n" + "=" * 70)
print("NO-MASK BASELINE — official test set")
print("=" * 70)
roi = pd.DataFrame(dict(y=t.label.values, p=t.prob.values))
unit(roi, "ROI")
for key, nm in (("lesion_key", "lesion"), ("breast_key", "breast"), ("patient_id", "patient")):
    if key == "breast_key" and side.isna().all():
        unit(None, "breast"); continue
    g = t.groupby(key).agg(p=("prob", "mean"), y=("label", "max"))
    unit(g, nm)

out = os.path.join(D, "nomask_twostream_officialsplit_test.csv")
t.rename(columns={"label": "true"})[
    ["img", "lesion_key", "patient_id", "breast_key", "true", "assessment", "prob"]
].to_csv(out, index=False)
print(f"\nsaved {os.path.basename(out)}  ({len(t)} rows)")
print("\nCompare against your headline (mask-weighted, official test):")
print("  ROI 0.8769 | lesion 0.9043 | breast 0.9016 | patient 0.9041")

wide-crop name match: pattern 0 matched 1696 / 1696 files
1696 regions | 892 patients | 1005 lesions | malignant 46.2%
official split: train 1098 | val 220 | test 378  (patient-disjoint)


/root/miniconda3/lib/python3.12/site-packages/sklearn/model_selection/_split.py:1036: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=6.
  warnings.warn(


cached 1696 tight + 1696 wide in 12s
helper heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5}

NO-MASK TWO-STREAM  (tight 512 GAP  ||  wide 384 GAP)  2048-d

  seed 11
    ep  1  val-AUC 0.7356 *
    ep  2  val-AUC 0.7748 *
    ep  3  val-AUC 0.7559
    ep  4  val-AUC 0.7908 *
    ep  5  val-AUC 0.8179 *
    ep  6  val-AUC 0.8316 *
    ep  7  val-AUC 0.8706 *
    ep  8  val-AUC 0.8715 *
    ep  9  val-AUC 0.8515
    ep 10  val-AUC 0.8894 *
    ep 11  val-AUC 0.8716
    ep 12  val-AUC 0.8754
    ep 13  val-AUC 0.8715
    ep 14  val-AUC 0.8639
    ep 15  val-AUC 0.8687
    ep 16  val-AUC 0.8629
    ep 17  val-AUC 0.8862
    early stop
  seed 11: best val 0.8894 | TEST ROI AUC 0.8571

  15.75 M params | 21 min
  global threshold fitted on val: 0.51

NO-MASK BASELINE — official test set
  ROI      n=378  AUC 0.8571   acc  78.3%  sens 0.789  spec 0.779  missed 31
  lesion   n=223  AUC 0.8816   acc  82.5%  sens 0.793  spec 0.846  missed 18
  breast   -- not derivable from filenames,

## F · Audits


**`IL2` cell 5** —   
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 9 — AUDIT OF THE AGGREGATION CHAIN.  No results, only checks.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
for c in ("side","view"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int)
d["a"]=pd.to_numeric(d.assessment,errors="coerce").fillna(4).astype(int).clip(0,5)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)

print("="*80); print("1. KEY HYGIENE"); print("="*80)
print("  side values   :", dict(d["side"].value_counts(dropna=False)))
print("  view values   :", dict(d["view"].value_counts(dropna=False)))
print("  side NaN %d | view NaN %d | assessment NaN %d"
      % (d["side"].isna().sum(), d["view"].isna().sum(),
         pd.to_numeric(d.assessment,errors='coerce').isna().sum()))
print("  crop key unique:", d["_k"].is_unique)

print("\n"+"="*80); print("2. UNIT COUNTS"); print("="*80)
for sp in ("train","test"):
    t=d[d.sp==sp]
    print("  %-5s  ROI %4d | lesion %4d | breast %4d | patient %4d | mammograms %4d"
          % (sp,len(t),t.lesion_key.nunique(),t.breast_key.nunique(),
             t.patient_id.nunique(),t.full_path.nunique() if "full_path" in t else -1))
tr,te=set(d[d.sp=="train"].patient_id),set(d[d.sp=="test"].patient_id)
print("  patient overlap %d | lesion_key overlap %d | breast overlap %d"
      % (len(tr&te),
         len(set(d[d.sp=='train'].lesion_key)&set(d[d.sp=='test'].lesion_key)),
         len(set(d[d.sp=='train'].breast_key)&set(d[d.sp=='test'].breast_key))))

print("\n"+"="*80); print("3. VIEWS PER LESION"); print("="*80)
g=d.groupby("lesion_key")
vc=g["view"].apply(lambda s: tuple(sorted(set(str(x).upper()[:3] for x in s))))
print("  view combinations per lesion:", dict(pd.Series(vc).value_counts()))
print("  lesions with >2 crops:", int((g.size()>2).sum()))
print("  lesions whose crops disagree on LABEL   :", int((g.y.nunique()>1).sum()))
print("  lesions whose crops disagree on BI-RADS :", int((g.a.nunique()>1).sum()))
print("  breasts containing >1 lesion :", int((d.groupby('breast_key').lesion_key.nunique()>1).sum()))
print("  patients with both breasts   :", int((d.groupby('patient_id').breast_key.nunique()>1).sum()))
print("  breasts mixing benign+malignant lesions:",
      int((d.groupby('breast_key').y.nunique()>1).sum()))
print("  patients mixing benign+malignant       :",
      int((d.groupby('patient_id').y.nunique()>1).sum()))

print("\n"+"="*80); print("4. HOW MUCH IS DECIDED BY THE BI-RADS 5 RULE ALONE"); print("="*80)
p=pd.read_csv(os.path.join(D,"cv_mass_twostream_officialsplit_oof.csv"))
p["_k"]=p["img"].map(stem)
m=d.merge(p[["_k","prob"]].drop_duplicates("_k"),on="_k").query("sp=='test'")
TAU={"ROI":{0:.515,1:.44,2:.44,3:.495,4:.455,5:.01},
     "LESION":{0:.52,1:.455,2:.455,3:.49,4:.465,5:.01}}
for nm,key in (("ROI",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")):
    if key is None:
        u=m.rename(columns={"prob":"pp"})[["pp","y","a"]]
    else:
        gg=m.groupby(key); u=gg.agg(pp=("prob","mean"),y=("y","max"),a=("a","max"))
    tau=TAU["ROI"] if nm=="ROI" else TAU["LESION"]
    forced=(u.a==5)
    yh=(u.pp>=u.a.map(lambda z: tau.get(int(z),0.45))).astype(int)
    acc_all=(yh==u.y).mean()
    acc_rest=(yh[~forced]==u.y[~forced]).mean() if (~forced).sum() else float("nan")
    print("  %-8s n=%-5d  BI-RADS 5 units %3d (%4.1f%%)  of which truly malignant %3d (%.0f%%)"
          % (nm,len(u),int(forced.sum()),100*forced.mean(),
             int(u.y[forced].sum()), 100*u.y[forced].mean() if forced.sum() else 0))
    print("           accuracy all %.1f%%   |   accuracy excluding BI-RADS 5 units %.1f%%"
          % (100*acc_all,100*acc_rest))
print("\n  If BI-RADS 5 units are a small share AND mostly truly malignant, the rule is")
print("  clinically correct rather than a shortcut. If accuracy collapses once they are")
print("  removed, most of the gain is deference to the radiologist and must be disclosed.")

1. KEY HYGIENE
  side values   : {'RIGHT': np.int64(879), 'LEFT': np.int64(817)}
  view values   : {'MLO': np.int64(912), 'CC': np.int64(784)}
  side NaN 0 | view NaN 0 | assessment NaN 0
  crop key unique: True

2. UNIT COUNTS
  train  ROI 1318 | lesion  782 | breast  722 | patient  691 | mammograms 1231
  test   ROI  378 | lesion  223 | breast  210 | patient  201 | mammograms  361
  patient overlap 0 | lesion_key overlap 0 | breast overlap 0

3. VIEWS PER LESION
  view combinations per lesion: {('CC', 'MLO'): np.int64(691), ('MLO',): np.int64(221), ('CC',): np.int64(93)}
  lesions with >2 crops: 0
  lesions whose crops disagree on LABEL   : 3
  lesions whose crops disagree on BI-RADS : 36
  breasts containing >1 lesion : 48
  patients with both breasts   : 40
  breasts mixing benign+malignant lesions: 5
  patients mixing benign+malignant       : 18

4. HOW MUCH IS DECIDED BY THE BI-RADS 5 RULE ALONE
  ROI      n=378    BI-RADS 5 units  75 (19.8%)  of which truly malignant  70 (93%)
 

**`IL2` cell 8** —   
<sub>1 output block(s) preserved</sub>


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 10 — IS THE TRAINING-SIDE PREDICTION FILE GENUINELY OUT-OF-FOLD?
#   Four independent diagnostics. No GPU, ~5 seconds.
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, brier_score_loss
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
d["y"]=d.label.astype(int)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
B=d.set_index("_k")

def load(fn):
    m=pd.read_csv(os.path.join(D,fn)); m["_k"]=m["img"].map(stem)
    m=m[m["_k"].isin(B.index)].drop_duplicates("_k")
    return m.set_index("_k")["prob"]

TR=load("cv_mass_twostream_officialtrain_oof.csv")
TE=load("cv_mass_twostream_officialsplit_oof.csv")

print("="*80); print("DIAGNOSTIC 1 — AUC LEVEL"); print("="*80)
for nm,s in (("train-side",TR),("test-side",TE)):
    y=B.loc[s.index,"y"].values
    print("  %-12s n=%4d   AUC %.4f" % (nm,len(s),roc_auc_score(y,s.values)))
print("  reference: an in-sample DenseNet fit typically scores AUC 0.98-0.999")
print("             an out-of-fold fit scores close to the test AUC")

print("\n"+"="*80); print("DIAGNOSTIC 2 — HOW EXTREME ARE THE PROBABILITIES?"); print("="*80)
print("  %-12s %-10s %-10s %-10s %s" % ("","<0.05","<0.10","| >0.90",">0.95"))
for nm,s in (("train-side",TR),("test-side",TE)):
    v=s.values
    print("  %-12s %-10.1f%% %-10.1f%% %-10.1f%% %.1f%%"
          % (nm,100*(v<.05).mean(),100*(v<.10).mean(),100*(v>.90).mean(),100*(v>.95).mean()))
print("  in-sample predictions collapse to the extremes (typically >70% outside 0.1-0.9).")
print("  if the two rows look alike, the training file is out-of-fold.")

print("\n"+"="*80); print("DIAGNOSTIC 3 — ACCURACY AND CALIBRATION AT 0.5"); print("="*80)
for nm,s in (("train-side",TR),("test-side",TE)):
    y=B.loc[s.index,"y"].values; v=s.values
    print("  %-12s acc@0.5 %.3f   Brier %.4f   mean prob: malignant %.3f | benign %.3f"
          % (nm,((v>=.5).astype(int)==y).mean(),brier_score_loss(y,v),
             v[y==1].mean(),v[y==0].mean()))
print("  in-sample accuracy@0.5 is normally 0.95-1.00 and Brier below 0.05")

print("\n"+"="*80); print("DIAGNOSTIC 4 — DOES A FOLD STRUCTURE EXIST INSIDE OFFICIAL TRAIN?"); print("="*80)
rc=[c for c in d.columns if c.startswith("role_of")]
if not rc:
    print("  no role_of* columns found — cannot confirm the fold structure from the csv")
else:
    trmask=d.sp.eq("train")
    vals=[set(d.loc[trmask & d[c].eq("val"),"_k"]) for c in sorted(rc)]
    union=set().union(*vals); overlap=sum(len(a&b) for i,a in enumerate(vals) for b in vals[i+1:])
    print("  role columns      : %s" % sorted(rc))
    print("  val-set sizes     : %s   (sum %d)" % ([len(v) for v in vals],sum(len(v) for v in vals)))
    print("  official train n  : %d" % int(trmask.sum()))
    print("  val sets overlap  : %d   (0 means they partition cleanly)" % overlap)
    print("  union covers train: %s" % ("YES" if union==set(d.loc[trmask,'_k']) else "NO"))
    if union==set(d.loc[trmask,"_k"]) and overlap==0:
        print("\n  per-fold AUC of the train-side predictions (each fold scored by the model")
        print("  that held it out — these should all be similar, none near 0.99):")
        for c in sorted(rc):
            k=set(d.loc[trmask & d[c].eq("val"),"_k"]) & set(TR.index)
            k=list(k); y=B.loc[k,"y"].values
            if len(set(y))>1:
                print("     %-10s n=%3d  AUC %.4f" % (c,len(k),roc_auc_score(y,TR.loc[k].values)))

print("\n"+"="*80); print("VERDICT"); print("="*80)
ytr=B.loc[TR.index,"y"].values; yte=B.loc[TE.index,"y"].values
a_tr,a_te=roc_auc_score(ytr,TR.values),roc_auc_score(yte,TE.values)
acc_tr=((TR.values>=.5).astype(int)==ytr).mean()
if a_tr>0.97 or acc_tr>0.94:
    print("  *** LOOKS IN-SAMPLE *** train AUC %.4f, acc@0.5 %.3f" % (a_tr,acc_tr))
    print("  Regenerate out-of-fold training predictions before fitting thresholds.")
else:
    print("  OUT-OF-FOLD CONFIRMED.  train AUC %.4f vs test %.4f (gap %+.4f), acc@0.5 %.3f"
          % (a_tr,a_te,a_tr-a_te,acc_tr))
    print("  Too low to be in-sample. The thresholds were fitted on honest held-out")
    print("  predictions, and your procedure is sound as described.")

DIAGNOSTIC 1 — AUC LEVEL
  train-side   n=1318   AUC 0.8974
  test-side    n= 378   AUC 0.8769
  reference: an in-sample DenseNet fit typically scores AUC 0.98-0.999
             an out-of-fold fit scores close to the test AUC

DIAGNOSTIC 2 — HOW EXTREME ARE THE PROBABILITIES?
               <0.05      <0.10      | >0.90    >0.95
  train-side   0.0       % 0.7       % 1.0       % 0.1%
  test-side    0.0       % 0.0       % 1.1       % 0.0%
  in-sample predictions collapse to the extremes (typically >70% outside 0.1-0.9).
  if the two rows look alike, the training file is out-of-fold.

DIAGNOSTIC 3 — ACCURACY AND CALIBRATION AT 0.5
  train-side   acc@0.5 0.827   Brier 0.1521   mean prob: malignant 0.644 | benign 0.363
  test-side    acc@0.5 0.807   Brier 0.1636   mean prob: malignant 0.629 | benign 0.382
  in-sample accuracy@0.5 is normally 0.95-1.00 and Brier below 0.05

DIAGNOSTIC 4 — DOES A FOLD STRUCTURE EXIST INSIDE OFFICIAL TRAIN?
  role columns      : ['role_of0', 'role_of1', 'ro

**`IL2` cell 11** —   
<sub>1 output block(s) preserved</sub>


In [2]:
# ══════════════════════════════════════════════════════════════════════
#  DATA INTEGRITY AUDIT  (hardened)
#  ROIs per mammogram | split safety | crop sanity | missing abnormalities
#  CPU, ~30 s. Writes nothing.
# ══════════════════════════════════════════════════════════════════════
import os, re, numpy as np, pandas as pd, cv2
cv2.setNumThreads(0)
D = "/root/autodl-tmp/CBIS"

d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d = d.drop(columns=[c for c in ["side", "view", "abn", "pid_f", "mammo", "breast"]
                    if c in d.columns])
b = d["img"].astype(str).map(os.path.basename)
print("sample filenames:")
for x in b.head(5): print("   ", x)

# --- parse patient / side / view / abnormality id ----------------------
PAT = r"(P[_-]\d+)[_-](LEFT|RIGHT)[_-](CC|MLO)(?:[_-](\d+))?"
ex = b.str.extract(PAT, flags=re.I, expand=True)
ex.columns = ["pid_f", "side", "view", "abn"]
for col in ["pid_f", "side", "view"]:
    ex[col] = ex[col].str.upper()
d = pd.concat([d, ex], axis=1)

print("\nparsed  side %d/%d | view %d/%d | abnormality-id %d/%d"
      % (d.side.notna().sum(), len(d), d.view.notna().sum(), len(d),
         d.abn.notna().sum(), len(d)))
if d.view.isna().any():
    print("  UNPARSED examples:", list(b[d.view.isna()].head(5)))
    print("  -> the regex does not match these; paste them to me")

d["mammo"]  = d.pid_f.fillna("?") + "_" + d.side.fillna("?") + "_" + d.view.fillna("?")
d["breast"] = d.pid_f.fillna("?") + "_" + d.side.fillna("?")

# --- Q1: ROIs per full mammogram --------------------------------------
r_per_m = d.groupby("mammo").size()
print("\n" + "=" * 66)
print("Q1  ROIs per full mammogram")
print("=" * 66)
print("  distinct mammograms   :", len(r_per_m))
print("  ROIs per mammogram    :", r_per_m.value_counts().sort_index().to_dict())
multi = r_per_m[r_per_m > 1]
print("  mammograms with >1 ROI: %d (%.1f%%)"
      % (len(multi), 100.0 * len(multi) / max(len(r_per_m), 1)))
print("\n  lesions per breast    :",
      d.groupby("breast")["lesion_key"].nunique().value_counts().sort_index().to_dict())
print("  lesions per patient   :",
      d.groupby("patient_id")["lesion_key"].nunique().value_counts().sort_index().to_dict())
print("  views per lesion      :",
      d.groupby("lesion_key")["view"]
       .apply(lambda s: "+".join(sorted(set(s.dropna()))) or "none")
       .value_counts().to_dict())

# --- Q2: does anything straddle the official split? -------------------
sp = d["official_split"].astype(str).str.lower().str.strip()
d["_split"] = np.where(sp.str.contains("test"), "test", "train")
print("\n" + "=" * 66)
print("Q2  does anything straddle the official train/test line?")
print("=" * 66)
for key in ["mammo", "breast", "lesion_key", "patient_id"]:
    n = int((d.groupby(key)["_split"].nunique() > 1).sum())
    print("  %-12s straddling : %d   %s" % (key, n, "OK" if n == 0 else "<<< PROBLEM"))

# --- Q3: is any "crop" actually a binary mask? ------------------------
print("\n" + "=" * 66)
print("Q3  crop sanity — is any 'crop' actually a mask?")
print("=" * 66)
rows = []
for p in d["img"].astype(str):
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if im is None:
        rows.append((p, -1, -1, -1, -1.0))
    else:
        rows.append((p, im.shape[0], im.shape[1], int(np.unique(im).size),
                     float(((im == 0) | (im == 255)).mean())))
c = pd.DataFrame(rows, columns=["img", "h", "w", "nuniq", "extreme"])
good = c[c.h > 0]
print("  unreadable files          : %d" % int((c.h < 0).sum()))
if len(good):
    lm = good[(good.nuniq <= 4) | (good.extreme > 0.95)]
    print("  crops that look like masks: %d  %s"
          % (len(lm), "OK" if len(lm) == 0 else "<<< INSPECT"))
    if len(lm):
        print(lm.assign(img=lm.img.map(os.path.basename)).head(10).to_string(index=False))
    print("  intensity levels per crop : median %d   (real crop = many, mask = 2)"
          % int(good.nuniq.median()))
    print("  crop size   h %d-%d   w %d-%d"
          % (good.h.min(), good.h.max(), good.w.min(), good.w.max()))
else:
    print("  <<< NO readable images at all — the 'img' paths are wrong")
    print("      sample path:", d['img'].iloc[0])

# --- Q4: any missing abnormalities? -----------------------------------
print("\n" + "=" * 66)
print("Q4  abnormality numbering — any gaps?")
print("=" * 66)
a = d.dropna(subset=["abn"]).copy()
if len(a):
    a["abn"] = pd.to_numeric(a["abn"], errors="coerce")
    a = a.dropna(subset=["abn"])
    g = a.groupby("mammo")["abn"].agg(mx="max", nu="nunique")
    gaps = g[g.mx != g.nu]
    print("  mammograms whose ids are not 1..N : %d  %s"
          % (len(gaps), "OK — no missing ROIs" if len(gaps) == 0 else "<<< ROIs absent"))
    if len(gaps): print(gaps.head(10).to_string())
else:
    print("  no abnormality id in filenames — cannot check")

sample filenames:
    3f87f40c5b125dcdee76d4d579f26b59_img.png
    93ece87e3fe98bd44f544b48f211a02f_img.png
    e6f2f4c7eda2376602e48cd1fbb109be_img.png
    4764a2b76b326c83c8fa12fdf7b2bf7b_img.png
    431825da8e5a6298c05fb42a44812cf9_img.png

parsed  side 0/1696 | view 0/1696 | abnormality-id 0/1696
  UNPARSED examples: ['3f87f40c5b125dcdee76d4d579f26b59_img.png', '93ece87e3fe98bd44f544b48f211a02f_img.png', 'e6f2f4c7eda2376602e48cd1fbb109be_img.png', '4764a2b76b326c83c8fa12fdf7b2bf7b_img.png', '431825da8e5a6298c05fb42a44812cf9_img.png']
  -> the regex does not match these; paste them to me

Q1  ROIs per full mammogram
  distinct mammograms   : 1
  ROIs per mammogram    : {1696: 1}
  mammograms with >1 ROI: 1 (100.0%)

  lesions per breast    : {1005: 1}
  lesions per patient   : {1: 821, 2: 44, 3: 21, 4: 3, 5: 1, 7: 1, 9: 1}
  views per lesion      : {'none': 1005}

Q2  does anything straddle the official train/test line?
  mammo        straddling : 1   <<< PROBLEM
  breast       stra

## G · Small checks and helpers


**`IL2` cell 6** — import os, numpy as np, pandas as pd  
<sub>1 output block(s) preserved</sub>


In [2]:
import os, numpy as np, pandas as pd
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
for c in ("side","view"):
    if c not in d.columns:
        fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
        d=d.merge(fx[["_k",c]].drop_duplicates("_k"),on="_k",how="left")
p=pd.read_csv(os.path.join(D,"cv_mass_twostream_officialsplit_oof.csv")); p["_k"]=p["img"].map(stem)
m=d.merge(p[["_k","prob"]].drop_duplicates("_k"),on="_k")
m["y"]=m.label.astype(int); m["a"]=pd.to_numeric(m.assessment,errors="coerce").fillna(4).astype(int)
m["breast_key"]=m.patient_id.astype(str)+"_"+m["side"].astype(str)

cand=(m.groupby("patient_id").agg(nb=("breast_key","nunique"),nl=("lesion_key","nunique"),
                                  ny=("y","nunique")).query("nb>1 or nl>1").index)
for pid in list(cand)[:2]:
    t=m[m.patient_id==pid].sort_values(["side","lesion_key","view"])
    print("\n"+"="*70); print("PATIENT %s" % pid); print("="*70)
    for _,r in t.iterrows():
        print("   %-6s %-4s %-28s prob %.3f  label %d  BI-RADS %d"
              % (r["side"],r["view"],r.lesion_key,r.prob,r.y,r.a))
    print("   -> lesion  : %s" % {k:round(v,3) for k,v in t.groupby('lesion_key').prob.mean().items()})
    print("   -> breast  : %s" % {k:round(v,3) for k,v in t.groupby('breast_key').prob.mean().items()})
    print("   -> patient : %.3f   label %d   BI-RADS(max) %d"
          % (t.prob.mean(), t.y.max(), t.a.max()))


PATIENT P_00116
   RIGHT  CC   P_00116_RIGHT_1              prob 0.403  label 1  BI-RADS 5
   RIGHT  MLO  P_00116_RIGHT_1              prob 0.419  label 1  BI-RADS 5
   RIGHT  CC   P_00116_RIGHT_2              prob 0.919  label 1  BI-RADS 5
   RIGHT  MLO  P_00116_RIGHT_2              prob 0.922  label 1  BI-RADS 5
   -> lesion  : {'P_00116_RIGHT_1': 0.411, 'P_00116_RIGHT_2': 0.92}
   -> breast  : {'P_00116_RIGHT': 0.666}
   -> patient : 0.666   label 1   BI-RADS(max) 5

PATIENT P_00173
   LEFT   CC   P_00173_LEFT_1               prob 0.361  label 0  BI-RADS 3
   LEFT   MLO  P_00173_LEFT_1               prob 0.177  label 0  BI-RADS 3
   RIGHT  CC   P_00173_RIGHT_1              prob 0.192  label 0  BI-RADS 3
   RIGHT  MLO  P_00173_RIGHT_1              prob 0.475  label 0  BI-RADS 3
   RIGHT  CC   P_00173_RIGHT_2              prob 0.171  label 0  BI-RADS 3
   RIGHT  MLO  P_00173_RIGHT_2              prob 0.333  label 0  BI-RADS 3
   -> lesion  : {'P_00173_LEFT_1': 0.269, 'P_00173_RIGHT_1

**`IL2` cell 7** — import json, pandas as pd, os  
<sub>1 output block(s) preserved</sub>


In [3]:
import json, pandas as pd, os
D="/root/autodl-tmp/CBIS"
print(open(os.path.join(D,"FINAL_floor090.json")).read()[:0])   # touch to confirm it exists
off = json.load(open(os.path.join(D,"FINAL_floor090.json")))
cv  = json.load(open(os.path.join(D,"FINAL_cv_and_ladder.json")))
grid= pd.read_csv(os.path.join(D,"operating_grid_official.csv"))

print("="*76); print("AUTHORITATIVE RESULTS — everything else in this notebook is history")
print("="*76)
print("\nOFFICIAL TEST, floor 0.90, mean aggregation, Novelty 1 + Novelty 2")
print(grid[(grid.system=="S2")&(grid.floor==0.90)]
        [["unit","n","auc","acc","sens","spec","fn"]].to_string(index=False))
print("\nNOVELTY 1 LADDER (official test AUC)")
for k,v in cv["ladder"].items(): print("   %-46s %.4f" % (k,v))
print("\n5-FOLD CV, floor 0.90")
for k,v in cv["cv"].items():
    print("   %-8s AUC %.4f  acc %.4f  sens %.3f  spec %.3f"
          % (k,v["auc"],v["acc"],v["sens"],v["spec"]))


AUTHORITATIVE RESULTS — everything else in this notebook is history

OFFICIAL TEST, floor 0.90, mean aggregation, Novelty 1 + Novelty 2
   unit   n      auc      acc     sens     spec  fn
  IMAGE 378 0.876903 0.809524 0.877551 0.766234  18
 LESION 223 0.904327 0.856502 0.885057 0.838235  10
 BREAST 210 0.901553 0.847619 0.882353 0.824000  10
PATIENT 201 0.904057 0.850746 0.882353 0.827586  10

NOVELTY 1 LADDER (official test AUC)
   baseline  plain DenseNet-121|IMAGE             0.7994
   baseline  plain DenseNet-121|LESION            0.8092
   baseline  plain DenseNet-121|BREAST            0.7970
   baseline  plain DenseNet-121|PATIENT           0.7960
   N1a       + mask-weighted pooling|IMAGE        0.8480
   N1a       + mask-weighted pooling|LESION       0.8715
   N1a       + mask-weighted pooling|BREAST       0.8647
   N1a       + mask-weighted pooling|PATIENT      0.8651
   N1b       + two-stream  [FINAL]|IMAGE          0.8769
   N1b       + two-stream  [FINAL]|LESION         0.

**`IL2` cell 12** — import os, pandas as pd  
<sub>1 output block(s) preserved</sub>


In [3]:
import os, pandas as pd
D = "/root/autodl-tmp/CBIS"

u = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
print("unified_folds_mass.csv  columns:"); print("  ", list(u.columns))
print("\n  sample lesion_key :", list(u.lesion_key.astype(str).head(6)))
print("  sample patient_id :", list(u.patient_id.astype(str).head(4)))
print("  sample img        :", list(u.img.astype(str).head(2)))

f = pd.read_csv(os.path.join(D, "cbis_mass_fixed.csv"))
print("\ncbis_mass_fixed.csv  %d rows  columns:" % len(f)); print("  ", list(f.columns))
print(f.head(3).to_string())

unified_folds_mass.csv  columns:
   ['patient_id', 'density', 'side', 'view', 'lesion', 'abn_type', 'calc_type', 'calc_dist', 'assessment', 'pathology', 'subtlety', 'full_path', 'cropped image file path', 'mask_path', 'official_split', 'mass_shape', 'mass_margins', 'mask_series', 'full_series', 'img', 'msk', 'source', 'label', 'split', 'lesion_key', 'fold', 'role_f0', 'role_f1', 'role_f2', 'role_f3', 'role_f4', 'oof_dice', 'oof_dice_tta', 'pred', 'role_cv0', 'role_cv1', 'role_cv2', 'role_cv3', 'role_cv4', 'role_of0', 'role_of1', 'role_of2', 'role_of3', 'role_of4', 'oof_dice_official']

  sample lesion_key : ['P_00016_LEFT_1', 'P_00016_LEFT_1', 'P_00017_LEFT_1', 'P_00017_LEFT_1', 'P_00032_RIGHT_1', 'P_00032_RIGHT_1']
  sample patient_id : ['P_00016', 'P_00016', 'P_00017', 'P_00017']
  sample img        : ['/root/autodl-tmp/CBIS/crops_fixed_mass/3f87f40c5b125dcdee76d4d579f26b59_img.png', '/root/autodl-tmp/CBIS/crops_fixed_mass/93ece87e3fe98bd44f544b48f211a02f_img.png']

cbis_mass_fixed.c